<a href="https://colab.research.google.com/github/almo-intellect/visu-predict/blob/main/VISU_Traffic_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nota da Propriedade Intelectual

Em conformidade com os direitos de propriedade intelectual e padrões de confidencialidade, esteja ciente de que os arquivos compartilhados com você por e-mail ou qualquer outro meio estão sujeitos à proteção sob as leis de propriedade intelectual. Esses arquivos podem conter informações proprietárias, segredos comerciais ou material protegido por direitos autorais de propriedade exclusiva da Almo Intellect.

Enfatizamos que esses arquivos são para sua referência e utilizados exclusivamente no contexto das tarefas ou responsabilidades que lhe foram atribuídas. Eles não devem ser distribuídos, transmitidos ou compartilhados com quaisquer outros indivíduos ou entidades sem autorização explícita da Almo Intellect.

A sua cooperação na manutenção da confidencialidade das informações contidas nestes ficheiros é crucial para preservar os nossos direitos de propriedade intelectual e manter a confidencialidade dos dados sensíveis.

Caso tenha alguma dúvida ou necessite de maiores esclarecimentos sobre o manuseio desses arquivos, sinta-se à vontade para contactar-nos.

Obrigado pela sua compreensão e estrita adesão a estas medidas de confidencialidade.

In compliance with intellectual property rights and confidentiality standards, please be aware that files shared with you via email or any other means are subject to protection under intellectual property laws. These files may contain proprietary information, trade secrets or copyrighted material exclusively owned by Almo Intellect.

We emphasize that these files are for your reference and used solely in the context of the tasks or responsibilities assigned to you. They must not be distributed, transmitted or shared with any other individuals or entities without explicit permission from Almo Intellect.

Your cooperation in maintaining the confidentiality of the information contained in these files is crucial to preserving our intellectual property rights and maintaining the confidentiality of sensitive data.

If you have any questions or require further clarification regarding the handling of these files, please feel free to contact us.

Thank you for your understanding and strict adherence to these confidentiality measures.

# VISU Traffic Transformer Model

In [42]:
from google.colab import drive

import os
if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')
else:
  print("Google Drive is already mounted at /content/drive")


!ls "/content/drive"

Google Drive is already mounted at /content/drive


## Imports

In [43]:
"""
Traffic Prediction with Transformers
Changelog:
- Structure with clear separation of concerns
- Type hints
- Mixed precision training for improved performance
- Gradient accumulation for effective larger batch sizes
"""

## @title VISU Traffic Transformer Model #TODO: Uncomment

from IPython import get_ipython
from IPython.display import display
# %%
# Install required libraries
# !pip install torch torchvision torchaudio pandas numpy scikit-learn matplotlib seaborn holidays statsmodels optuna torch_geometric pytz #pmdarima
!pip install torch pandas numpy scikit-learn matplotlib seaborn holidays statsmodels optuna torch_geometric pytz #torchvision torchaudio  #pmdarima

import os
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional, Union, Any, Callable
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.nn import TransformerEncoder, TransformerEncoderLayer
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import math
import torch.optim as _optim
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.gridspec import GridSpec
import pytz
from datetime import datetime
from scipy import stats
import warnings
import gc

# Optional imports with error handling
try:
    import holidays
except ImportError:
    warnings.warn("holidays package not found, holiday features will be limited")
    holidays = None

# Try to import necessary mixed precision components
try:
    import torch.cuda.amp
    from torch.cuda.amp import autocast, GradScaler
    AMP_AVAILABLE = True
    # Check if newer API with device_type is available
    try:
        # Try creating a GradScaler with device_type
        test_scaler = GradScaler(device_type='cuda')
        del test_scaler
        DEVICE_TYPE_SUPPORTED = True
    except TypeError:
        # Older PyTorch version without device_type support
        DEVICE_TYPE_SUPPORTED = False
except ImportError:
    warnings.warn("Mixed precision training not available (torch.cuda.amp not available)")
    AMP_AVAILABLE = False
    DEVICE_TYPE_SUPPORTED = False

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Benchmarking Imports
try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    STATSMODELS_AVAILABLE = True
except ImportError:
    warnings.warn("statsmodels not available, some benchmarks will be disabled")
    STATSMODELS_AVAILABLE = False

# Optuna for hyperparameter tuning
try:
    import optuna
    OPTUNA_AVAILABLE = True
except ImportError:
    warnings.warn("optuna not available, hyperparameter optimization will be disabled")
    OPTUNA_AVAILABLE = False

# PyTorch Geometric Imports
try:
    from torch_geometric.nn import GCNConv, GATConv
    TORCH_GEOMETRIC_AVAILABLE = True
except ImportError:
    warnings.warn("PyTorch Geometric not available, GNN functionality will be limited")
    TORCH_GEOMETRIC_AVAILABLE = False


<ipython-input-43-3833b3e24d6d>:61: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  test_scaler = GradScaler(device_type='cuda')


## GPU Memory & Performance Utilities

In [44]:

# =============================================================================
# GPU Memory & Performance Utilities
# =============================================================================

def get_gpu_memory_info():
    """Get GPU memory usage information"""
    if not torch.cuda.is_available():
        return {"error": "CUDA not available"}

    try:
        # Try to use nvidia-smi via subprocess if needed
        # But for simplicity, we'll use PyTorch's built-in functions
        device_count = torch.cuda.device_count()
        gpu_info = {}

        for i in range(device_count):
            total_memory = torch.cuda.get_device_properties(i).total_memory
            reserved_memory = torch.cuda.memory_reserved(i)
            allocated_memory = torch.cuda.memory_allocated(i)
            free_memory = total_memory - reserved_memory

            gpu_info[f"gpu_{i}"] = {
                "total_memory_GB": total_memory / 1e9,
                "reserved_memory_GB": reserved_memory / 1e9,
                "allocated_memory_GB": allocated_memory / 1e9,
                "free_memory_GB": free_memory / 1e9,
                "utilization_pct": (allocated_memory / total_memory) * 100
            }

        return gpu_info

    except Exception as e:
        return {"error": str(e)}

def find_optimal_batch_size(
    model: nn.Module,
    sample_input: torch.Tensor,
    sample_target: torch.Tensor,
    max_batch_size: int = 2048,
    start_batch: int = 32,
    device: str = 'cuda'
) -> int:
    """
    Find the optimal batch size for the model and GPU memory

    Args:
        model: The model to test
        sample_input: A sample input tensor
        sample_target: A sample target tensor
        max_batch_size: Maximum batch size to test
        start_batch: Starting batch size
        device: Device to test on

    Returns:
        Optimal batch size
    """
    if device == 'cpu' or not torch.cuda.is_available():
        return 64  # Default for CPU

    model = model.to(device)
    optimal_batch_size = start_batch

    # Try to clear some memory first
    torch.cuda.empty_cache()
    gc.collect()

    print("Finding optimal batch size for GPU...")
    try:
        # Get a copy of the sample tensors on the correct device
        sample_input = sample_input.to(device)
        sample_target = sample_target.to(device)

        for batch_size in [2**i for i in range(int(np.log2(start_batch)), int(np.log2(max_batch_size))+1)]:
            try:
                # Try to process a batch of this size
                input_batch = sample_input.repeat(batch_size, 1, 1)
                target_batch = sample_target.repeat(batch_size, 1, 1)

                # Forward and backward pass
                if DEVICE_TYPE_SUPPORTED:
                    with autocast(device_type='cuda'):
                        output = model(input_batch)
                        loss = nn.MSELoss()(output, target_batch)
                else:
                    with autocast():
                        output = model(input_batch)
                        loss = nn.MSELoss()(output, target_batch)

                loss.backward()

                # If we got here without an OOM error, update optimal batch size
                optimal_batch_size = batch_size

                # Clean up to prevent memory accumulation
                del input_batch, target_batch, output, loss
                torch.cuda.empty_cache()

                print(f"  Successfully tested batch size: {batch_size}")

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"  OOM at batch size: {batch_size}")
                    break
                else:
                    raise

    except Exception as e:
        print(f"Error while finding optimal batch size: {e}")
        return 64  # Fallback to default

    finally:
        # Clean up
        torch.cuda.empty_cache()
        gc.collect()

    # Return a slightly smaller batch size to be safe
    return max(int(optimal_batch_size * 0.8), start_batch)

# Simplified GPU monitoring - no longer using contextmanager to avoid import issues
def start_gpu_memory_monitor(config, interval=10):
    """Start GPU memory monitoring - returns a flag to control monitoring"""
    if not config.monitor_gpu_usage or not torch.cuda.is_available():
        return None

    # Import time here to ensure it's available
    import time

    # Flag to control monitoring
    stop_monitoring = [False]

    # Function that will run in a separate thread
    def monitor_memory_usage():
        print("Starting GPU memory monitoring...")
        start_time = time.time()

        while not stop_monitoring[0]:
            try:
                memory_info = get_gpu_memory_info()

                # Print memory usage for each GPU
                for gpu_id, gpu_data in memory_info.items():
                    if isinstance(gpu_data, dict) and 'error' not in gpu_data:
                        print(f"{gpu_id.upper()}: "
                              f"Used {gpu_data['allocated_memory_GB']:.2f}/{gpu_data['total_memory_GB']:.2f} GB "
                              f"({gpu_data['utilization_pct']:.1f}%)")

                # Also monitor CPU memory if psutil is available
                try:
                    import psutil
                    process = psutil.Process(os.getpid())
                    print(f"CPU Memory: {process.memory_info().rss / 1e9:.2f} GB")
                except ImportError:
                    pass

                time.sleep(interval)
            except Exception as e:
                print(f"Error in GPU monitoring: {e}")
                time.sleep(interval)

    try:
        # Start monitoring in a separate thread
        import threading
        monitor_thread = threading.Thread(target=monitor_memory_usage)
        monitor_thread.daemon = True  # Daemon thread will exit when main thread exits
        monitor_thread.start()

        return stop_monitoring
    except Exception as e:
        print(f"Failed to start GPU monitoring: {e}")
        return None

def stop_gpu_memory_monitor(stop_flag):
    """Stop GPU memory monitoring by setting the stop flag"""
    if stop_flag is not None:
        stop_flag[0] = True


## Config Module

In [45]:

# =============================================================================
# Config Module
# =============================================================================

@dataclass
class TrainingConfig:
    """Configuration class for training parameters with enhanced validation"""
    base_output_dir: str
    batch_size: int = 512
    seq_length: int = 24
    pred_length: int = 24
    num_epochs: int = 50
    patience: int = 10
    learning_rate: float = 0.001
    hidden_dim: int = 512
    num_layers: int = 6
    num_heads: int = 8
    dropout: float = 0.1
    ff_dim_multiplier: int = 4
    activation: str = 'relu'
    scaler_type: str = 'minmax'
    optimizer_type: str = 'adamw'
    loss_function: str = 'mse'
    use_time_features: bool = True
    use_holiday_feature: bool = False
    holiday_country_code: str = 'US'
    use_weather_feature: bool = False
    gradient_clip: Optional[float] = 1.0
    scheduler_type: Optional[str] = 'plateau'
    scheduler_patience: int = 5
    scheduler_factor: float = 0.5
    step_scheduler_step_size: int = 10
    step_scheduler_gamma: float = 0.1
    use_lagged_features: bool = False
    num_lags: int = 24
    decoder_type: str = 'linear'
    use_spatial_features: bool = False
    spatial_feature_dim: int = 2
    use_gnn_pre_transformer: bool = False
    gnn_type: str = 'gcn'
    use_quantile_regression: bool = False
    quantiles: List[float] = None
    optuna_trials: Optional[int] = 10
    warmup_epochs: int = 5
    use_mixed_precision: bool = True
    accumulation_steps: int = 32
    num_workers: int = 2  # Reduced from 4 to 2 based on system warning
    pin_memory: bool = True
    find_optimal_batch_size: bool = True
    monitor_gpu_usage: bool = True

    # Directories (to be set after initialization)
    input_dir: Optional[str] = None
    output_dir: Optional[str] = None
    model_dir: Optional[str] = None
    results_dir: Optional[str] = None

    def __post_init__(self):
        if self.quantiles is None:
            self.quantiles = [0.1, 0.5, 0.9]

        # Enhanced check for divisibility with better warning messages
        if self.num_heads > 0:
            if self.hidden_dim % self.num_heads != 0:
                original_hidden_dim = self.hidden_dim
                # Adjust hidden_dim to be divisible by num_heads
                self.hidden_dim = (self.hidden_dim // self.num_heads) * self.num_heads

                # If adjustment resulted in zero, set it to num_heads
                if self.hidden_dim == 0:
                    self.hidden_dim = self.num_heads

                warnings.warn(f"Adjusted hidden_dim from {original_hidden_dim} to {self.hidden_dim} to ensure divisibility by num_heads={self.num_heads}")

            # Also ensure hidden_dim is even for positional encoding
            if self.hidden_dim % 2 != 0:
                original_hidden_dim = self.hidden_dim
                self.hidden_dim += 1  # Make it even by adding 1
                warnings.warn(f"Adjusted hidden_dim from {original_hidden_dim} to {self.hidden_dim} to ensure it's even for positional encoding")

        # Check for mixed precision availability
        if self.use_mixed_precision and not AMP_AVAILABLE:
            self.use_mixed_precision = False
            warnings.warn("Mixed precision requested but not available, disabled")

        # Check for GNN availability
        if self.use_gnn_pre_transformer and not TORCH_GEOMETRIC_AVAILABLE:
            self.use_gnn_pre_transformer = False
            warnings.warn("GNN pre-transformer requested but PyTorch Geometric not available, disabled")

        # Adjust workers based on system capabilities
        import multiprocessing
        max_workers = multiprocessing.cpu_count()
        if self.num_workers > max_workers:
            warnings.warn(f"Reducing num_workers from {self.num_workers} to {max_workers} based on system capabilities")
            self.num_workers = max_workers


def ensure_compatible_dimensions(model_params, config):
    """
    Ensures that hidden_dim is divisible by num_heads and is even
    for compatibility with the MultiheadAttention module

    Args:
        model_params: Dictionary of model parameters
        config: Training configuration object

    Returns:
        Updated model_params and config
    """
    # Check if hidden_dim is divisible by num_heads
    if model_params['hidden_dim'] % model_params['num_heads'] != 0:
        # Adjust to nearest value divisible by num_heads
        adjusted_hidden_dim = (model_params['hidden_dim'] // model_params['num_heads']) * model_params['num_heads']
        # Ensure it's not zero
        if adjusted_hidden_dim == 0:
            adjusted_hidden_dim = model_params['num_heads']

        print(f"Warning: Adjusting hidden_dim from {model_params['hidden_dim']} to {adjusted_hidden_dim} to ensure divisibility by num_heads={model_params['num_heads']}")
        model_params['hidden_dim'] = adjusted_hidden_dim
        config.hidden_dim = adjusted_hidden_dim

    # Also ensure hidden_dim is even for positional encoding
    if model_params['hidden_dim'] % 2 != 0:
        adjusted_hidden_dim = model_params['hidden_dim'] + 1
        print(f"Warning: Adjusting hidden_dim from {model_params['hidden_dim']} to {adjusted_hidden_dim} to ensure it's even for positional encoding")
        model_params['hidden_dim'] = adjusted_hidden_dim
        config.hidden_dim = adjusted_hidden_dim

    return model_params, config


def load_config(config_path: str = 'config.yaml') -> TrainingConfig:
    """Load configuration from YAML file and create TrainingConfig object"""
    try:
        with open(config_path, 'r') as f:
            config_dict = yaml.safe_load(f)
        return TrainingConfig(**config_dict)
    except (FileNotFoundError, yaml.YAMLError) as e:
        warnings.warn(f"Error loading config from {config_path}: {e}. Using default configuration.")
        return TrainingConfig(base_output_dir='./output')


def setup_directories(config: TrainingConfig) -> Tuple[str, str, str, str]:
    """Sets up input, output, model, and results directories with timestamped folders."""
    timestamp = get_maputo_timestamp()

    output_dir = os.path.join(config.base_output_dir, f"Transformers_Output_{timestamp}")
    input_dir = os.path.join(config.base_output_dir, f"Transformers_Input")
    model_dir = os.path.join(output_dir, f"Models_{timestamp}")
    results_dir = os.path.join(output_dir, f"Results_{timestamp}")

    os.makedirs(input_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)

    # Update config with directory paths
    config.input_dir = input_dir
    config.output_dir = output_dir
    config.model_dir = model_dir
    config.results_dir = results_dir

    return input_dir, output_dir, model_dir, results_dir


def get_maputo_timestamp() -> str:
    """Returns current timestamp in Maputo timezone."""
    maputo_tz = pytz.timezone('Africa/Maputo')
    return datetime.now(maputo_tz).strftime("%Y%m%d_%H%M%S")



## Data Module

In [46]:

# =============================================================================
# Data Module
# =============================================================================

class TrafficDataset(Dataset):
    """Enhanced Traffic Dataset with proper typing and improved feature handling"""

    def __init__(
        self,
        data: np.ndarray,
        timestamps: Optional[pd.DatetimeIndex] = None,
        sequence_length: int = 24,
        prediction_window: int = 1,
        config: Optional[TrainingConfig] = None
    ):
        """
        Traffic Dataset with configurable features including lagged, spatial, and weather.

        Args:
            data: Sensor data array
            timestamps: Timestamps for time-based features
            sequence_length: Input sequence length
            prediction_window: Prediction window
            config: Configuration object with feature flags
        """
        self.data = data
        self.seq_length = sequence_length
        self.pred_window = prediction_window
        self.timestamps = timestamps
        self.config = config or TrainingConfig(base_output_dir="./output")

        self.features_list = []

        if self.config.use_time_features and timestamps is not None:
            self.create_time_features(timestamps)
            self.features_list.extend(['hour', 'dayofweek', 'weekofyear', 'month'])

        if self.config.use_holiday_feature and timestamps is not None:
            self.create_holiday_feature(timestamps)
            self.features_list.append('is_holiday')

        if self.config.use_weather_feature and timestamps is not None:
            self.create_weather_feature(timestamps)
            self.features_list.append('weather_condition')

        if self.config.use_lagged_features:
            self.create_lagged_features()
            self.features_list.extend([f'lag_{i+1}' for i in range(self.config.num_lags)])

        if self.config.use_spatial_features:
            self.create_spatial_features()
            self.features_list.extend([f'spatial_feature_{i+1}' for i in range(self.config.spatial_feature_dim)])

    def create_time_features(self, timestamps: pd.DatetimeIndex) -> None:
        """Create normalized time-based features"""
        times = pd.to_datetime(timestamps)
        self.time_features = np.stack([
            times.hour.values / 23.0,
            times.dayofweek.values / 6.0,
            (times.dayofyear // 7).values / 51.0,
            times.month.values / 11.0
        ], axis=1)

    def create_holiday_feature(self, timestamps: pd.DatetimeIndex) -> None:
        """Create holiday indicator feature"""
        if holidays is None:
            # Create dummy holiday feature if holidays package is not available
            self.holiday_feature = np.zeros((len(timestamps), 1))
            warnings.warn("holidays package not available, using dummy holiday feature")
            return

        dates = pd.to_datetime(timestamps).date
        country_code = self.config.holiday_country_code
        try:
            country_holidays = holidays.CountryHoliday(country_code, years=set(d.year for d in dates))
        except KeyError:
            warnings.warn(f"Country code '{country_code}' not recognized. Using US holidays instead.")
            country_holidays = holidays.CountryHoliday('US', years=set(d.year for d in dates))

        self.holiday_feature = np.array([(1 if d in country_holidays else 0) for d in dates], dtype=float).reshape(-1, 1)

    def create_weather_feature(self, timestamps: pd.DatetimeIndex) -> None:
        """Placeholder for weather feature creation."""
        num_samples = len(timestamps)
        # Simulate weather conditions (0: clear, 1: rain, 2: snow)
        weather_conditions = np.random.randint(0, 3, num_samples).reshape(-1, 1) / 2.0
        self.weather_feature = weather_conditions
        warnings.warn("Using simulated weather features. Replace with actual weather data integration.")

    def create_lagged_features(self) -> None:
        """Creates lagged features from the sensor data itself."""
        num_lags = self.config.num_lags
        lagged_features = []

        for i in range(1, num_lags + 1):
            lagged_data = np.roll(self.data, shift=i, axis=0)
            lagged_data[:i] = np.nan  # Fill first 'i' rows with NaN
            lagged_features.append(lagged_data)

        self.lagged_features = np.concatenate(lagged_features, axis=1)
        # Handle NaN values with zero filling
        self.lagged_features = np.nan_to_num(self.lagged_features, nan=0.0)

    def create_spatial_features(self) -> None:
        """
        Placeholder for spatial feature creation using random embeddings.
        """
        num_sensors = self.data.shape[1]
        spatial_feature_dim = self.config.spatial_feature_dim
        # Simulate spatial features (random embeddings)
        spatial_features = np.random.rand(num_sensors, spatial_feature_dim)
        self.spatial_features = np.tile(spatial_features, (len(self.data), 1))

    def get_adjacency_matrix(self) -> torch.Tensor:
        """
        Get adjacency matrix for graph neural network.
        """
        num_sensors = self.data.shape[1]
        # Example: Fully connected graph (replace with actual adjacency matrix)
        adjacency_matrix = torch.ones(num_sensors, num_sensors) - torch.eye(num_sensors)
        return adjacency_matrix

    def __len__(self) -> int:
        return len(self.data) - self.seq_length - self.pred_window + 1

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        x_data = self.data[idx:idx+self.seq_length]
        x_features = []

        if hasattr(self, 'time_features') and self.config.use_time_features:
            x_features.append(self.time_features[idx:idx+self.seq_length])

        if hasattr(self, 'holiday_feature') and self.config.use_holiday_feature:
            x_features.append(self.holiday_feature[idx:idx+self.seq_length])

        if hasattr(self, 'weather_feature') and self.config.use_weather_feature:
            x_features.append(self.weather_feature[idx:idx+self.seq_length])

        if hasattr(self, 'lagged_features') and self.config.use_lagged_features:
            x_lagged = self.lagged_features[idx:idx+self.seq_length]
            x_features.append(x_lagged)

        if hasattr(self, 'spatial_features') and self.config.use_spatial_features:
            x_spatial = self.spatial_features[idx:idx+self.seq_length]
            x_features.append(x_spatial)

        if x_features:
            x = np.concatenate([x_data] + x_features, axis=1)
        else:
            x = x_data

        y = self.data[idx+self.seq_length:idx+self.seq_length+self.pred_window]
        return torch.FloatTensor(x), torch.FloatTensor(y)


def prepare_data(
    df: pd.DataFrame,
    config: TrainingConfig
) -> Tuple[np.ndarray, pd.DatetimeIndex, Any, int]:
    """
    Prepare and preprocess data for training

    Args:
        df: Input dataframe with timestamp index
        config: Training configuration

    Returns:
        Tuple of:
        - Normalized data
        - Timestamps
        - Scaler object
        - Number of features
    """
    # Handle null values
    df.replace(0.0, np.nan, inplace=True)
    df.ffill(inplace=True)
    df.bfill(inplace=True)

    timestamps = df.index
    sensor_data = df.values
    num_features = sensor_data.shape[1]

    # Data Scaling
    if config.scaler_type == 'minmax':
        scaler = MinMaxScaler()
    elif config.scaler_type == 'standard':
        scaler = StandardScaler()
    elif config.scaler_type == 'robust':
        scaler = RobustScaler()
    else:
        warnings.warn(f"Invalid scaler type: {config.scaler_type}, using MinMaxScaler")
        scaler = MinMaxScaler()

    data_normalized = scaler.fit_transform(sensor_data)

    return data_normalized, timestamps, scaler, num_features



## Model Module

In [47]:

# =============================================================================
# Model Module
# =============================================================================

class CustomTransformerEncoderLayer(nn.TransformerEncoderLayer):
    """Modified Transformer Encoder Layer to capture attention weights."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.attn_weights = None

    def _sa_block(self, x, attn_mask, key_padding_mask, is_causal=False):
        x, weights = self.self_attn(
            x, x, x,
            attn_mask=attn_mask,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            is_causal=is_causal
        )
        self.attn_weights = weights.detach()
        return self.dropout1(x)


class PositionalEncoding(nn.Module):
    """
    Positional Encoding module with enhanced dimension checking and error handling.
    Ensures d_model is even and properly applies positional encoding to the input.
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Ensure d_model is even
        if d_model % 2 != 0:
            raise ValueError(f"d_model must be even for positional encoding, got {d_model}")

        # Explicitly initialize buffer with proper shape
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        # Compute div_term with precise checks
        div_term_indices = torch.arange(0, d_model, 2).float()
        try:
            div_term = torch.exp(div_term_indices * (-math.log(10000.0) / d_model))
        except Exception as e:
            raise ValueError(f"Error calculating div_term: {e}, d_model={d_model}")

        # Do additional validation
        if len(div_term) * 2 > d_model:
            raise ValueError(f"div_term length ({len(div_term)}) too large for d_model={d_model}")

        # Apply sin and cos with explicit shape checking
        try:
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
        except Exception as e:
            raise RuntimeError(f"Error in positional encoding calculation: {e}\n"
                              f"Shapes - position: {position.shape}, div_term: {div_term.shape}, "
                              f"pe: {pe.shape}, d_model: {d_model}")

        # Register as buffer (not a parameter)
        pe = pe.unsqueeze(0).transpose(0, 1)  # Shape: [max_len, 1, d_model]
        self.register_buffer('pe', pe)

        # For debugging
        self.d_model = d_model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape [seq_len, batch_size, d_model]

        Returns:
            Tensor with positional encoding added
        """
        # Validate input shape
        if x.size(-1) != self.d_model:
            raise ValueError(f"Input feature dimension {x.size(-1)} doesn't match "
                           f"positional encoding dimension {self.d_model}")

        # Add positional encoding with proper shape handling
        try:
            max_len = min(x.size(0), self.pe.size(0))
            x = x + self.pe[:max_len, :]
        except Exception as e:
            raise RuntimeError(f"Error adding positional encoding: {e}\n"
                              f"Shapes - x: {x.shape}, pe: {self.pe.shape}, "
                              f"x size(0): {x.size(0)}, pe size(0): {self.pe.size(0)}")

        return self.dropout(x)


class GCNEncoder(nn.Module):
    """Graph Convolutional Network Encoder"""
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_layers: int = 2,
        dropout: float = 0.1,
        gnn_type: str = 'gcn'
    ):
        super(GCNEncoder, self).__init__()

        if not TORCH_GEOMETRIC_AVAILABLE:
            raise ImportError("PyTorch Geometric is required for GCNEncoder")

        self.layers = nn.ModuleList()

        if gnn_type == 'gcn':
            gnn_layer = GCNConv
        elif gnn_type == 'gat':
            gnn_layer = GATConv
        else:
            raise ValueError(f"Invalid GNN type: {gnn_type}. Choose 'gcn' or 'gat'.")

        self.layers.append(gnn_layer(input_dim, hidden_dim))  # First GNN layer

        for _ in range(num_layers - 1):
            self.layers.append(gnn_layer(hidden_dim, hidden_dim))  # Subsequent GNN layers

        self.dropout = nn.Dropout(dropout)
        self.gnn_type = gnn_type

    def forward(self, x: torch.Tensor, adjacency_matrix: torch.Tensor) -> torch.Tensor:
        for i, layer in enumerate(self.layers):
            if self.gnn_type == 'gat' and i == 0:  # GAT needs explicit head specification in first layer
                x = layer(x, adjacency_matrix, heads=8)  # Example heads for GAT
            else:
                x = layer(x, adjacency_matrix)

            if i < len(self.layers) - 1:
                x = F.relu(x)  # Activation after each layer except last
                x = self.dropout(x)

        return x


class TrafficTransformer(nn.Module):
    """
    Traffic Transformer Model with configurable components
    """
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_layers: int,
        num_heads: int,
        num_features: int,  # New: explicitly passing number of features
        dropout: float = 0.1,
        ff_dim_multiplier: int = 4,
        activation: str = 'relu',
        decoder_type: str = 'linear',
        use_gnn_pre_transformer: bool = False,
        spatial_feature_dim: int = 0,
        gnn_type: str = 'gcn',
        pred_len: int = 1
    ):
        """
        Initialize Traffic Transformer Model

        Args:
            input_dim: Input feature dimension
            hidden_dim: Hidden dimension for transformer
            num_layers: Number of transformer layers
            num_heads: Number of attention heads
            num_features: Number of features in output (spatial dimension)
            dropout: Dropout rate
            ff_dim_multiplier: Multiplier for feedforward dimension
            activation: Activation function ('relu' or 'gelu')
            decoder_type: Type of decoder ('linear' or 'mlp')
            use_gnn_pre_transformer: Whether to use GNN before transformer
            spatial_feature_dim: Dimension of spatial features (for GNN)
            gnn_type: Type of GNN ('gcn' or 'gat')
            pred_len: Prediction length (temporal dimension)
        """
        super(TrafficTransformer, self).__init__()

        # Validate dimensions before initialization
        if hidden_dim % num_heads != 0:
            raise ValueError(f"Hidden dimension {hidden_dim} must be divisible by number of attention heads {num_heads}")

        if hidden_dim % 2 != 0:
            raise ValueError(f"Hidden dimension {hidden_dim} must be even for positional encoding, got {hidden_dim}")

        self.attention_weights = None
        self.use_gnn_pre_transformer = use_gnn_pre_transformer
        self.pred_len = pred_len
        self.num_features = num_features  # Store number of features

        if use_gnn_pre_transformer:
            if not TORCH_GEOMETRIC_AVAILABLE:
                raise ImportError("PyTorch Geometric is required for GNN pre-transformer")

            self.gnn_encoder = GCNEncoder(
                spatial_feature_dim,
                hidden_dim,
                dropout=dropout,
                gnn_type=gnn_type
            )
            transformer_input_dim = hidden_dim
        else:
            self.embedding = nn.Linear(input_dim, hidden_dim)
            transformer_input_dim = hidden_dim

        self.pos_encoder = PositionalEncoding(hidden_dim, dropout)

        # Encoder Layers with configurable activation
        encoder_layers = [
            CustomTransformerEncoderLayer(
                hidden_dim,
                num_heads,
                hidden_dim * ff_dim_multiplier,
                dropout,
                batch_first=True,
                activation=activation
            ) for _ in range(num_layers)
        ]
        self.transformer = nn.ModuleList(encoder_layers)

        # Configurable Decoder
        if decoder_type == 'linear':
            self.decoder = nn.Linear(hidden_dim, num_features * pred_len)
        elif decoder_type == 'mlp':
            self.decoder = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, num_features * pred_len)
            )
        else:
            raise ValueError(f"Invalid decoder type: {decoder_type}. Choose 'linear' or 'mlp'.")

    def forward(self, src: torch.Tensor, adjacency_matrix: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Forward pass through the transformer model

        Args:
            src: Input tensor [batch_size, seq_len, feature_dim]
            adjacency_matrix: Optional adjacency matrix for GNN

        Returns:
            Output tensor [batch_size, pred_len, num_features]
        """
        if src.dim() == 2:
            src = src.unsqueeze(1)

        if self.use_gnn_pre_transformer:
            if adjacency_matrix is None:
                raise ValueError("adjacency_matrix is required when use_gnn_pre_transformer=True")

            # Extract spatial features from input
            spatial_features = src[:, :, -self.gnn_encoder.layers[0].in_channels:]
            src_temporal = src[:, :, :-self.gnn_encoder.layers[0].in_channels:]

            # Apply GCN to spatial features
            src_spatial_encoded = self.gnn_encoder(spatial_features.mean(dim=1), adjacency_matrix)

            # Concatenate GCN-encoded spatial features with temporal features
            src = torch.cat([
                src_temporal,
                src_spatial_encoded.unsqueeze(1).repeat(1, src.size(1), 1)
            ], dim=-1)
        else:
            src = self.embedding(src)  # Apply linear embedding if no GNN

        src = self.pos_encoder(src)

        # Apply transformer layers
        for i, layer in enumerate(self.transformer):
          src = layer(src)
          if i == len(self.transformer) - 1:  # Last layer
              # Store attention weights properly - take mean across batch dimension
              if hasattr(layer, 'attn_weights') and layer.attn_weights is not None:
                  self.attention_weights = layer.attn_weights.mean(dim=0) if layer.attn_weights.dim() > 2 else layer.attn_weights



        # Apply decoder to last timestep
        output = self.decoder(src[:, -1, :])

        # Reshape to [batch_size, pred_len, num_features]
        return output.view(-1, self.pred_len, self.num_features)


class PyTorchLSTMForecaster(nn.Module):
    """Improved LSTM-based model for time series forecasting"""
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_layers: int,
        seq_length: int,  # Added sequence length parameter
        pred_length: int,  # Added prediction length parameter
        dropout: float = 0.1,
        epochs: int = 100,
        batch_size: int = 32,
        learning_rate: float = 0.001,
        device: str = 'cpu'
    ):
        super(PyTorchLSTMForecaster, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.seq_length = seq_length
        self.pred_length = pred_length
        self.input_size = input_size
        self.output_size = output_size

        # LSTM with dropout
        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        # Decoder to map from hidden state to output
        self.decoder = nn.Linear(hidden_size, output_size * pred_length)

        # Training parameters
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.device = device
        self.history = {'loss': [], 'val_loss': []}  # Store training history

    def forward(self, x):
        """
        Forward pass through LSTM model

        Args:
            x: Input tensor of shape [batch_size, seq_length, input_size]

        Returns:
            Output tensor of shape [batch_size, pred_length, output_size]
        """
        # Initialize hidden state and cell state
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # Forward propagate LSTM
        out, _ = self.lstm(x, (h0, c0))

        # Decode the hidden state of the last time step
        out = self.decoder(out[:, -1, :])

        # Reshape to [batch_size, pred_length, output_size]
        out = out.view(-1, self.pred_length, self.output_size)

        return out

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        """
        Train the LSTM model on input data with proper sequence handling

        Args:
            X_train: Training input of shape [num_samples, seq_length, input_size]
            y_train: Training target of shape [num_samples, pred_length, output_size]
            X_val: Optional validation input
            y_val: Optional validation target
        """
        self.to(self.device)
        optimizer = optim.Adam(self.parameters(), lr=self.learning_rate)
        criterion = nn.MSELoss()

        # Convert numpy arrays to tensors if needed
        if not isinstance(X_train, torch.Tensor):
            X_train = torch.tensor(X_train, dtype=torch.float32)
        if not isinstance(y_train, torch.Tensor):
            y_train = torch.tensor(y_train, dtype=torch.float32)

        # Move to device
        X_train = X_train.to(self.device)
        y_train = y_train.to(self.device)

        # Prepare validation data if provided
        if X_val is not None and y_val is not None:
            if not isinstance(X_val, torch.Tensor):
                X_val = torch.tensor(X_val, dtype=torch.float32)
            if not isinstance(y_val, torch.Tensor):
                y_val = torch.tensor(y_val, dtype=torch.float32)

            X_val = X_val.to(self.device)
            y_val = y_val.to(self.device)

        # Create data loaders
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )

        # Early stopping
        best_loss = float('inf')
        patience = 5
        no_improve = 0

        for epoch in range(self.epochs):
            self.train()  # Set model to training mode
            total_loss = 0

            for X_batch, y_batch in train_loader:
                # Forward pass
                outputs = self(X_batch)
                loss = criterion(outputs, y_batch)

                # Backward and optimize
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            avg_train_loss = total_loss / len(train_loader)
            self.history['loss'].append(avg_train_loss)

            # Validation if data provided
            if X_val is not None and y_val is not None:
                self.eval()  # Set model to evaluation mode
                with torch.no_grad():
                    val_outputs = self(X_val)
                    val_loss = criterion(val_outputs, y_val)

                val_loss = val_loss.item()
                self.history['val_loss'].append(val_loss)

                # Early stopping check
                if val_loss < best_loss:
                    best_loss = val_loss
                    no_improve = 0
                else:
                    no_improve += 1

                if no_improve >= patience:
                    print(f'Early stopping at epoch {epoch+1}')
                    break

                print(f'Epoch [{epoch+1}/{self.epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}')
            else:
                print(f'Epoch [{epoch+1}/{self.epochs}], Loss: {avg_train_loss:.4f}')

    def predict(self, X):
        """
        Generate predictions with the trained model

        Args:
            X: Input data of shape [num_samples, seq_length, input_size]

        Returns:
            Predictions of shape [num_samples, pred_length, output_size]
        """
        self.eval()  # Set model to evaluation mode

        # Convert to tensor if needed
        if not isinstance(X, torch.Tensor):
            X = torch.tensor(X, dtype=torch.float32)

        # Move to device
        X = X.to(self.device)

        with torch.no_grad():
            predictions = self(X)

        return predictions.cpu().numpy()

# Custom learning rate scheduler with warmup
class CosineWarmupLR(_optim.lr_scheduler.LRScheduler):
    """Cosine annealing with warmup learning rate scheduler"""
    def __init__(
        self,
        optimizer: torch.optim.Optimizer,
        warmup_epochs: int,
        total_epochs: int,
        base_lr: float,
        warmup_lr: float = 0.0,
        last_epoch: int = -1
    ):
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = base_lr
        self.warmup_lr = warmup_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self) -> List[float]:
        if self.last_epoch < self.warmup_epochs:
            # Linear warmup phase
            alpha = self.last_epoch / self.warmup_epochs
            return [self.warmup_lr + (self.base_lr - self.warmup_lr) * alpha] * len(self.optimizer.param_groups)
        else:
            # Cosine annealing phase
            progress = float(self.last_epoch - self.warmup_epochs) / float(max(1, self.total_epochs - self.warmup_epochs))
            return [max(0.0, self.base_lr * 0.5 * (1.0 + math.cos(math.pi * progress)))] * len(self.optimizer.param_groups)



## Training Module

In [48]:

# =============================================================================
# Training Module
# =============================================================================

def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler: Optional[torch.optim.lr_scheduler._LRScheduler],
    criterion: Callable,
    config: TrainingConfig,
    device: str = 'cpu',
    adjacency_matrix: Optional[torch.Tensor] = None
) -> Tuple[nn.Module, List[float], List[float]]:
    """
    Train the traffic forecasting model with support for mixed precision and gradient accumulation

    Args:
        model: The model to train
        train_loader: DataLoader with training data
        val_loader: DataLoader with validation data
        optimizer: Optimizer for training
        scheduler: Learning rate scheduler
        criterion: Loss function
        config: Training configuration
        device: Device to train on ('cpu' or 'cuda')
        adjacency_matrix: Optional adjacency matrix for GNN

    Returns:
        Tuple of (trained model, training losses, validation losses)
    """
    num_epochs = config.num_epochs
    patience = config.patience
    best_loss = float('inf')
    no_improve = 0
    train_losses = []
    val_losses = []
    model_dir = getattr(config, 'model_dir', None)
    use_quantile_regression = config.use_quantile_regression
    loss_function_type = config.loss_function
    accumulation_steps = config.accumulation_steps

    # Setup for mixed precision training
    use_amp = config.use_mixed_precision and AMP_AVAILABLE and device != 'cpu'
    if use_amp:
        if DEVICE_TYPE_SUPPORTED:
            scaler = GradScaler(device_type='cuda')
        else:
            # Older PyTorch versions don't support device_type
            scaler = GradScaler()
    else:
        scaler = None

    # Log memory usage initially
    if torch.cuda.is_available() and device == 'cuda':
        print(f"Initial GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
        print(f"Initial GPU memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        optimizer.zero_grad()  # Zero gradients at the start of each epoch

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            # Mixed precision forward pass
            if use_amp:
                if DEVICE_TYPE_SUPPORTED:
                    with autocast(device_type='cuda'):
                        if config.use_spatial_features and config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)

                        if use_quantile_regression:
                            quantiles = config.quantiles
                            loss = quantile_loss(output, target, quantiles) / accumulation_steps
                        elif loss_function_type == 'hybrid':
                            loss = hybrid_loss(output, target) / accumulation_steps
                        else:
                            loss = criterion(output, target) / accumulation_steps
                else:
                    # Older PyTorch versions
                    with autocast():
                        if config.use_spatial_features and config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)

                        if use_quantile_regression:
                            quantiles = config.quantiles
                            loss = quantile_loss(output, target, quantiles) / accumulation_steps
                        elif loss_function_type == 'hybrid':
                            loss = hybrid_loss(output, target) / accumulation_steps
                        else:
                            loss = criterion(output, target) / accumulation_steps

                # Mixed precision backward pass
                scaler.scale(loss).backward()

                # Gradient accumulation - only step every accumulation_steps
                if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
                    if config.gradient_clip is not None:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)

                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
            else:
                # Standard precision training
                if config.use_spatial_features and config.use_gnn_pre_transformer:
                    output = model(data, adjacency_matrix.to(device))
                else:
                    output = model(data)

                if use_quantile_regression:
                    quantiles = config.quantiles
                    loss = quantile_loss(output, target, quantiles) / accumulation_steps
                elif loss_function_type == 'hybrid':
                    loss = hybrid_loss(output, target) / accumulation_steps
                else:
                    loss = criterion(output, target) / accumulation_steps

                # Standard backward pass
                loss.backward()

                # Gradient accumulation - only step every accumulation_steps
                if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
                    if config.gradient_clip is not None:
                        nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)

                    optimizer.step()
                    optimizer.zero_grad()

            # Track loss (use full loss for logging)
            train_loss += loss.item() * accumulation_steps

        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Report GPU memory usage after training epoch
        if torch.cuda.is_available() and device == 'cuda':
            print(f"GPU memory after training: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB / "
                  f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB "
                  f"({torch.cuda.memory_allocated(0) / torch.cuda.get_device_properties(0).total_memory * 100:.1f}%)")

        # Validation
        avg_val_loss, _ = evaluate_model(model, val_loader, criterion, device, config, adjacency_matrix)
        val_losses.append(avg_val_loss)

        # Early stopping check
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            no_improve = 0

            # Save best model if model_dir is available
            if model_dir is not None:
                try:
                    timestamp = get_maputo_timestamp()
                    model_path = os.path.join(model_dir, f'best_model_{timestamp}.pth')
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'loss': best_loss,
                        'config': {k: v for k, v in vars(config).items() if not k.startswith('_')}
                    }, model_path)
                except Exception as e:
                    print(f"Warning: Could not save model: {str(e)}")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

        # Update learning rate
        if config.scheduler_type == 'cosine_warmup':
            scheduler.step()
        elif scheduler and config.scheduler_type == 'plateau':
            scheduler.step(avg_val_loss)
        elif scheduler:  # Other scheduler types
            scheduler.step()

        print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')

    # Plot training history if results_dir is available
    if hasattr(config, 'results_dir') and config.results_dir is not None:
        try:
            plot_training_history(train_losses, val_losses, config.results_dir)
        except Exception as e:
            print(f"Warning: Could not plot training history: {str(e)}")

    return model, train_losses, val_losses


def evaluate_model(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: Callable,
    device: str = 'cpu',
    config: Optional[TrainingConfig] = None,
    adjacency_matrix: Optional[torch.Tensor] = None
) -> Tuple[float, Tuple[float, float, float, float]]:
    """
    Evaluate the model and calculate metrics

    Args:
        model: Model to evaluate
        dataloader: DataLoader with evaluation data
        criterion: Loss function
        device: Device to evaluate on
        config: Training configuration
        adjacency_matrix: Optional adjacency matrix for GNN

    Returns:
        Tuple of (average loss, (MAE, RMSE, R², MAPE))
    """
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    use_quantile_regression = config.use_quantile_regression if config else False
    loss_function_type = config.loss_function if config else 'mse'

    # Use mixed precision for evaluation if enabled
    use_amp = config and config.use_mixed_precision and AMP_AVAILABLE and device != 'cpu'

    with torch.no_grad():
        for data, target in dataloader:
            data, target = data.to(device), target.to(device)

            if use_amp:
                if DEVICE_TYPE_SUPPORTED:
                    with autocast(device_type='cuda'):
                        if config and config.use_spatial_features and config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)

                        if use_quantile_regression:
                            quantiles = config.quantiles
                            loss = quantile_loss(output, target, quantiles)
                        elif loss_function_type == 'hybrid':
                            loss = hybrid_loss(output, target)
                        else:
                            loss = criterion(output, target)
                else:
                    # Older PyTorch versions
                    with autocast():
                        if config and config.use_spatial_features and config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)

                        if use_quantile_regression:
                            quantiles = config.quantiles
                            loss = quantile_loss(output, target, quantiles)
                        elif loss_function_type == 'hybrid':
                            loss = hybrid_loss(output, target)
                        else:
                            loss = criterion(output, target)
            else:
                if config and config.use_spatial_features and config.use_gnn_pre_transformer:
                    output = model(data, adjacency_matrix.to(device))
                else:
                    output = model(data)

                if use_quantile_regression:
                    quantiles = config.quantiles
                    loss = quantile_loss(output, target, quantiles)
                elif loss_function_type == 'hybrid':
                    loss = hybrid_loss(output, target)
                else:
                    loss = criterion(output, target)

            total_loss += loss.item()
            all_preds.append(output.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    predictions = np.concatenate(all_preds)
    actuals = np.concatenate(all_targets)

    # Calculate metrics
    mae = mean_absolute_error(actuals.ravel(), predictions.ravel())
    rmse = np.sqrt(mean_squared_error(actuals.ravel(), predictions.ravel()))
    r2 = r2_score(actuals.ravel(), predictions.ravel())
    mape = robust_mape(actuals.ravel(), predictions.ravel())

    print(f'MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.2f}, MAPE: {mape:.2f}%')

    return total_loss / len(dataloader), (mae, rmse, r2, mape)


def predict(
    model: nn.Module,
    dataloader: DataLoader,
    scaler: Any,
    device: str = 'cpu',
    adjacency_matrix: Optional[torch.Tensor] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Make predictions using the trained model and inverse transform the results

    Args:
        model: Trained model
        dataloader: DataLoader with test data
        scaler: Scaler used for normalization
        device: Device to use for prediction
        adjacency_matrix: Optional adjacency matrix for GNN

    Returns:
        Tuple of (predictions, actuals) in original scale
    """
    model.eval()
    all_preds = []
    all_targets = []

    # Use AMP for prediction if available on GPU
    use_amp = hasattr(torch.cuda, 'amp') and device != 'cpu'

    with torch.no_grad():
        for data, target in dataloader:
            data, target = data.to(device), target.to(device)

            if use_amp:
                if DEVICE_TYPE_SUPPORTED:
                    with autocast(device_type='cuda'):
                        if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)
                else:
                    # Older PyTorch versions
                    with autocast():
                        if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)
            else:
                if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                    output = model(data, adjacency_matrix.to(device))
                else:
                    output = model(data)

            all_preds.append(output.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    predictions = np.concatenate(all_preds)
    actuals = np.concatenate(all_targets)

    # Reshape for inverse transformation
    num_samples, pred_window, num_features = predictions.shape
    predictions_2d = predictions.reshape(-1, num_features)
    actuals_2d = actuals.reshape(-1, num_features)

    # Inverse transform
    predictions_inv = scaler.inverse_transform(predictions_2d)
    actuals_inv = scaler.inverse_transform(actuals_2d)

    return predictions_inv, actuals_inv


def evaluate_baseline_model(predictions: np.ndarray, actuals: np.ndarray) -> Tuple[float, float, float, float]:
    """
    Evaluate baseline model predictions against actuals

    Args:
        predictions: Predicted values
        actuals: Actual values

    Returns:
        Tuple of (MAE, RMSE, R², MAPE)
    """
    mae = mean_absolute_error(actuals.ravel(), predictions.ravel())
    rmse = np.sqrt(mean_squared_error(actuals.ravel(), predictions.ravel()))
    r2 = r2_score(actuals.ravel(), predictions.ravel())
    mape = robust_mape(actuals.ravel(), predictions.ravel())
    return mae, rmse, r2, mape


def train_baseline_models(
    train_data: np.ndarray,
    test_data: np.ndarray,
    config: TrainingConfig,
    device: str,
    timestamps_train: Optional[pd.DatetimeIndex] = None,
    timestamps_test: Optional[pd.DatetimeIndex] = None
) -> Dict[str, List[Tuple[float, float, float, float]]]:
    """
    Train and evaluate baseline models with improved ARIMA and Exponential Smoothing
    """
    baseline_metrics = {'arima': [], 'exp_smoothing': [], 'lstm': []}

    if not STATSMODELS_AVAILABLE:
        warnings.warn("statsmodels not available, skipping ARIMA and ExponentialSmoothing baselines")
        baseline_metrics = {'lstm': []}

    # Create sequences for LSTM
    def create_sequences(data, seq_length, pred_length):
        X, y = [], []
        for i in range(len(data) - seq_length - pred_length + 1):
            X.append(data[i:i+seq_length])
            y.append(data[i+seq_length:i+seq_length+pred_length])
        return np.array(X), np.array(y)

    # Process only first 1-2 sensors to save time (SIGNIFICANT CHANGE)
    max_sensors = min(2, train_data.shape[1])
    print(f"Training baseline models for {max_sensors} sensors only to save time")

    # Create time features for exogenous variables if timestamps are available
    exog_train = None
    exog_test = None

    if timestamps_train is not None and timestamps_test is not None:
        # Create simplified exogenous variables (hour of day only)
        hour_train = timestamps_train.hour.values.reshape(-1, 1) / 23.0  # Normalize
        hour_test = timestamps_test.hour.values.reshape(-1, 1) / 23.0    # Normalize

        # Use simplified encoding to save time
        exog_train = hour_train
        exog_test = hour_test

        print(f"Using simplified exogenous variables for ARIMA")

    for sensor_idx in range(max_sensors):
        print(f"Training baseline models for sensor {sensor_idx+1}/{max_sensors}...")

        # Extract sensor data
        train_sensor_data = train_data[:, sensor_idx]
        test_sensor_data = test_data[:, sensor_idx]

        # ARIMA model with significant simplifications for speed
        if STATSMODELS_AVAILABLE:
            try:
                # Set a maximum timeout for ARIMA fitting (SIGNIFICANT CHANGE)
                import signal
                from contextlib import contextmanager

                class TimeoutException(Exception): pass

                @contextmanager
                def time_limit(seconds):
                    def signal_handler(signum, frame):
                        raise TimeoutException("Timed out!")
                    signal.signal(signal.SIGALRM, signal_handler)
                    signal.alarm(seconds)
                    try:
                        yield
                    finally:
                        signal.alarm(0)

                # Use simple ARIMA model with fixed parameters to save time
                try:
                    # Try with a very short timeout first
                    with time_limit(60):  # 60 seconds timeout
                        print("  Fitting simplified ARIMA(1,1,1) model with 60s timeout...")
                        # Use simpler model with fixed parameters
                        arima_model = ARIMA(
                            train_sensor_data[-500:],  # Use only last 500 points
                            order=(1, 1, 1),  # Very simple model
                            # No seasonal component for speed
                        ).fit()

                        # Generate predictions
                        arima_preds = arima_model.forecast(steps=len(test_sensor_data)).reshape(-1, 1)
                except TimeoutException:
                    print("  ARIMA timed out after 60s, using even simpler model")
                    # Fall back to extremely simple model
                    arima_preds = np.roll(test_sensor_data, 1).reshape(-1, 1)
                    arima_preds[0] = train_sensor_data[-1]  # Use last training value for first prediction

                # Evaluate ARIMA predictions
                arima_metrics = evaluate_baseline_model(
                    arima_preds,
                    test_sensor_data.reshape(-1, 1)
                )
                baseline_metrics['arima'].append(arima_metrics)
                print(f"  ARIMA - MAE: {arima_metrics[0]:.4f}, RMSE: {arima_metrics[1]:.4f}, R²: {arima_metrics[2]:.4f}")

            except Exception as e:
                warnings.warn(f"Failed to train ARIMA model for sensor {sensor_idx}: {e}")
                print(f"  Skipping ARIMA for this sensor due to error: {str(e)}")

        # Improved Exponential Smoothing model
        if STATSMODELS_AVAILABLE:
            try:
                # Try different ETS models and select the best one
                ets_models = [
                    {'trend': None, 'seasonal': None, 'seasonal_periods': None},
                    {'trend': 'add', 'seasonal': None, 'seasonal_periods': None},
                    {'trend': 'add', 'seasonal': 'add', 'seasonal_periods': 24},
                    {'trend': 'add', 'seasonal': 'mul', 'seasonal_periods': 24},
                    {'trend': 'mul', 'seasonal': 'add', 'seasonal_periods': 24},
                    {'trend': 'mul', 'seasonal': 'mul', 'seasonal_periods': 24},
                    {'trend': 'add', 'seasonal': 'add', 'seasonal_periods': 168},  # Weekly seasonality
                ]

                best_aic = float('inf')
                best_ets_params = None
                best_ets_model = None

                # Find validation split point for model selection
                val_size = min(int(len(train_sensor_data) * 0.2), 168)  # 20% or one week, whichever is smaller
                train_val_split = len(train_sensor_data) - val_size

                train_ets_data = train_sensor_data[:train_val_split]
                val_ets_data = train_sensor_data[train_val_split:]

                for params in ets_models:
                    try:
                        # Skip models that would cause errors
                        if params['seasonal'] and not params['seasonal_periods']:
                            continue

                        model = ExponentialSmoothing(
                            train_ets_data,
                            trend=params['trend'],
                            seasonal=params['seasonal'],
                            seasonal_periods=params['seasonal_periods'],
                            damped_trend=True if params['trend'] else None
                        )

                        model_fit = model.fit(optimized=True)

                        # Predict on validation set
                        val_preds = model_fit.forecast(steps=len(val_ets_data))

                        # Calculate validation error
                        val_mse = np.mean((val_ets_data - val_preds) ** 2)

                        # Use MSE for selection instead of AIC for more direct performance comparison
                        if val_mse < best_aic:
                            best_aic = val_mse
                            best_ets_params = params
                            best_ets_model = model_fit

                    except Exception as e:
                        print(f"  Error fitting ETS model with params {params}: {e}")

                if best_ets_model is None:
                    # Fallback to simple exponential smoothing
                    best_ets_model = ExponentialSmoothing(
                        train_sensor_data,
                        trend=None,
                        seasonal=None
                    ).fit()
                    best_ets_params = {'trend': None, 'seasonal': None, 'seasonal_periods': None}
                    print("  Using simple exponential smoothing as fallback")
                else:
                    print(f"  Best ETS model: {best_ets_params} with validation MSE: {best_aic:.6f}")

                    # Refit on the entire training data with the best parameters
                    best_model = ExponentialSmoothing(
                        train_sensor_data,
                        trend=best_ets_params['trend'],
                        seasonal=best_ets_params['seasonal'],
                        seasonal_periods=best_ets_params['seasonal_periods'],
                        damped_trend=True if best_ets_params['trend'] else None
                    ).fit(optimized=True)

                # Implement rolling forecast
                n_test = len(test_sensor_data)
                forecast_horizon = min(24, n_test)  # Forecast 24 steps ahead or less

                # Initialize array for predictions
                ets_preds = np.zeros(n_test)

                # Initial model
                model = best_model

                for i in range(0, n_test, forecast_horizon):
                    end_idx = min(i + forecast_horizon, n_test)
                    steps = end_idx - i

                    forecast = model.forecast(steps=steps)
                    ets_preds[i:end_idx] = forecast

                    # Update model if not at the end
                    if end_idx < n_test:
                        updated_train = np.concatenate([train_sensor_data, test_sensor_data[:end_idx]])

                        model = ExponentialSmoothing(
                            updated_train,
                            trend=best_ets_params['trend'],
                            seasonal=best_ets_params['seasonal'],
                            seasonal_periods=best_ets_params['seasonal_periods'],
                            damped_trend=True if best_ets_params['trend'] else None
                        ).fit(optimized=True)

                # Evaluate exponential smoothing predictions
                exp_smoothing_metrics = evaluate_baseline_model(
                    ets_preds.reshape(-1, 1),
                    test_sensor_data.reshape(-1, 1)
                )
                baseline_metrics['exp_smoothing'].append(exp_smoothing_metrics)
                print(f"  Improved Exp. Smoothing - MAE: {exp_smoothing_metrics[0]:.4f}, RMSE: {exp_smoothing_metrics[1]:.4f}, R²: {exp_smoothing_metrics[2]:.4f}")

            except Exception as e:
                warnings.warn(f"Failed to train improved Exponential Smoothing model for sensor {sensor_idx}: {e}")
                import traceback
                traceback.print_exc()

        # LSTM model (kept as in the original implementation)
        try:
            # Create sequences for LSTM with the same sequence length as transformer
            sensor_data = train_data[:, sensor_idx:sensor_idx+1]  # Keep dimension for input_size=1
            X_train, y_train = create_sequences(
                sensor_data,
                config.seq_length,
                config.pred_length
            )

            test_sensor_data = test_data[:, sensor_idx:sensor_idx+1]
            X_test, y_test = create_sequences(
                test_sensor_data,
                config.seq_length,
                config.pred_length
            )

            # Early stopping condition
            if len(X_train) < 10 or len(X_test) < 10:
                warnings.warn(f"Insufficient data for LSTM with sequences for sensor {sensor_idx}")
                continue

            # Initialize and train LSTM model
            lstm_model = PyTorchLSTMForecaster(
                input_size=1,
                hidden_size=64,  # Smaller hidden size for faster training
                output_size=1,
                num_layers=2,
                seq_length=config.seq_length,
                pred_length=config.pred_length,
                dropout=0.2,
                epochs=min(config.num_epochs, 30),  # Limit epochs for speed
                batch_size=min(config.batch_size, 128),
                learning_rate=config.learning_rate,
                device=device
            )

            # Train LSTM
            lstm_model.fit(X_train, y_train)

            # Predict
            lstm_preds = lstm_model.predict(X_test)

            # Evaluate - flatten the predictions and targets
            lstm_metrics = evaluate_baseline_model(
                lstm_preds.reshape(-1, 1),
                y_test.reshape(-1, 1)
            )
            baseline_metrics['lstm'].append(lstm_metrics)
            print(f"  LSTM - MAE: {lstm_metrics[0]:.4f}, RMSE: {lstm_metrics[1]:.4f}, R²: {lstm_metrics[2]:.4f}")

        except Exception as e:
            warnings.warn(f"Failed to train LSTM model for sensor {sensor_idx}: {e}")
            import traceback
            traceback.print_exc()

    return baseline_metrics


# --- Loss Functions ---
def quantile_loss(output: torch.Tensor, target: torch.Tensor, quantiles: List[float]) -> torch.Tensor:
    """
    Quantile Loss function for prediction intervals

    Args:
        output: Model output with shape [batch, pred_len, num_quantiles]
        target: Target values
        quantiles: List of quantiles

    Returns:
        Quantile loss value
    """
    losses = []
    for i, q in enumerate(quantiles):
        errors = target - output[:, :, i]
        losses.append(torch.max((q-1) * errors, q * errors).mean())
    loss = torch.sum(torch.stack(losses))
    return loss


def hybrid_loss(output: torch.Tensor, target: torch.Tensor, alpha: float = 0.5) -> torch.Tensor:
    """
    Hybrid Loss function: Weighted combination of MSE and MAE

    Args:
        output: Model output
        target: Target values
        alpha: Weight for MSE component (1-alpha for MAE)

    Returns:
        Hybrid loss value
    """
    mse_loss = nn.MSELoss()(output, target)
    mae_loss = nn.L1Loss()(output, target)
    loss = alpha * mse_loss + (1 - alpha) * mae_loss
    return loss


def robust_mape(y_true: np.ndarray, y_pred: np.ndarray, epsilon: float = 1e-8) -> float:
    """
    Robust MAPE to handle division by zero and near-zero values

    Args:
        y_true: True values
        y_pred: Predicted values
        epsilon: Small value to prevent division by zero

    Returns:
        MAPE value as percentage
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    if not np.any(mask):
        return np.nan  # Or 0, or another appropriate value if all y_true are zero
    y_true_masked = y_true[mask]
    y_pred_masked = y_pred[mask]
    return np.mean(np.abs((y_true_masked - y_pred_masked) / (y_true_masked + epsilon))) * 100


## Visualization Module

In [49]:


# =============================================================================
# Visualization Module
# =============================================================================

def plot_attention_weights(
    model: nn.Module,
    seq_length: int,
    results_dir: str,
    config: Optional[TrainingConfig] = None  # Add config parameter
) -> None:
    """
    Plots attention weights with timestamped filename and model details
    """
    plt.figure(figsize=(12, 10))  # Larger figure to accommodate details

    # Use 4/5 of the figure for the attention plot
    plt.subplot(5, 1, (1, 4))

    if not hasattr(model, 'attention_weights') or model.attention_weights is None:
        plt.text(0.5, 0.5, "No attention weights captured", ha='center', va='center', fontsize=14)
        plt.title('Attention Weights - Not Available')
    else:
        try:
            # Get attention weights and ensure proper shape
            weights = model.attention_weights

            # Convert to numpy if it's a tensor
            if isinstance(weights, torch.Tensor):
                weights = weights.cpu().detach().numpy()

            # Handle different possible shapes
            if len(weights.shape) == 3:  # [batch, seq, seq]
                weights = weights[0] if weights.shape[0] == 1 else np.mean(weights, axis=0)
            elif len(weights.shape) == 2 and weights.shape[1] == 1:  # [seq, 1] shape
                weights = np.tile(weights, (1, seq_length))

            # Create heatmap
            sns.heatmap(weights, cmap='viridis',
                        xticklabels=range(1, weights.shape[1] + 1),
                        yticklabels=range(1, weights.shape[0] + 1))
            plt.title('Attention Weights - Last Layer', fontsize=16)
            plt.xlabel('Key Positions')
            plt.ylabel('Query Positions')

        except Exception as e:
            # Provide error information in the plot
            plt.clf()
            plt.text(0.5, 0.5, f"Error plotting attention weights: {str(e)}\n"
                               f"Shape: {getattr(model.attention_weights, 'shape', 'unknown')}",
                     ha='center', va='center', wrap=True)
            plt.title('Attention Visualization Error')

    # Add model details at the bottom 1/5 of the figure
    model_details = ""
    if config:
        model_details = (
            f"Model: Transformer | Layers: {config.num_layers} | Heads: {config.num_heads} | "
            f"Hidden Dim: {config.hidden_dim} | Seq Length: {seq_length} | "
            f"Prediction Length: {config.pred_length} | Dropout: {config.dropout} | "
            f"Generated: {get_maputo_timestamp()}"
        )

    plt.subplot(5, 1, 5)
    plt.axis('off')
    plt.text(0.01, 0.5, model_details, wrap=True, fontsize=9)

    # Save with timestamp
    timestamp = get_maputo_timestamp()
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f'attention_heatmap_{timestamp}.png'), dpi=300)
    plt.close()


def plot_predictions_vs_actual(
    actuals: np.ndarray,
    predictions: np.ndarray,
    sensor_index: int,
    fold: int,
    results_dir: str,
    pred_len: int,
    config: Optional[TrainingConfig] = None  # Add config parameter
) -> None:
    """
    Plots actual vs predicted traffic flow with enhanced details
    """
    # Calculate R-squared specifically for this plot (subset of data)
    r2 = r2_score(actuals[:200, sensor_index], predictions[:200, sensor_index])

    plt.figure(figsize=(14, 8))

    # Main plot
    plt.subplot(4, 1, (1, 3))  # Use 3/4 of the figure for the main plot
    plt.plot(actuals[:200, sensor_index], label='Actual', linewidth=2)
    plt.plot(predictions[:200, sensor_index], label='Predicted', linewidth=2, alpha=0.8)

    # Enhanced title with R-squared
    plt.title(f'Fold {fold+1} - Actual vs Predicted Traffic Flow (Sensor {sensor_index+1})\n'
              f'R² = {r2:.4f}', fontsize=14, fontweight='bold')

    plt.xlabel('Time Steps')
    plt.ylabel('Traffic Flow')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)

    # Add model details and run information at the bottom
    model_details = ""
    if config:
        model_details = (
            f"Model: Transformer | Layers: {config.num_layers} | Heads: {config.num_heads} | "
            f"Hidden Dim: {config.hidden_dim} | Prediction Length: {pred_len} | "
            f"Learning Rate: {config.learning_rate} | Optimizer: {config.optimizer_type} | "
            f"Loss: {config.loss_function} | Time Features: {config.use_time_features} | "
            f"Batch Size: {config.batch_size} | Generated: {get_maputo_timestamp()}"
        )

    # Use the bottom 1/4 of the figure for model details
    plt.subplot(4, 1, 4)
    plt.axis('off')
    plt.text(0.01, 0.5, model_details, wrap=True, fontsize=9)

    # Save with timestamp
    timestamp = get_maputo_timestamp()
    filename = f'predictions_fold{fold+1}_sensor{sensor_index+1}_step{pred_len}_{timestamp}.png'
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, filename), dpi=300)
    plt.close()


def plot_training_history(
    train_losses: List[float],
    val_losses: List[float],
    results_dir: str,
    config: Optional[TrainingConfig] = None  # Add config parameter
) -> None:
    """
    Plots training and validation loss history with enhanced details
    """
    plt.figure(figsize=(12, 8))

    # Use 3/4 of the figure for the main plot
    plt.subplot(4, 1, (1, 3))
    plt.plot(train_losses, label='Training Loss', linewidth=2)
    plt.plot(val_losses, label='Validation Loss', linewidth=2)
    plt.title('Training History', fontsize=16, fontweight='bold')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Calculate improvement metrics
    if len(train_losses) > 0:
        initial_loss = train_losses[0]
        final_loss = train_losses[-1]
        best_val_loss = min(val_losses) if val_losses else 0

        improvement = 100 * (initial_loss - final_loss) / initial_loss if initial_loss > 0 else 0

        # Add text annotation for improvement
        plt.annotate(
            f'Training loss reduced by {improvement:.2f}%',
            xy=(len(train_losses) * 0.6, (initial_loss + final_loss) / 2),
            xytext=(len(train_losses) * 0.4, final_loss + (initial_loss - final_loss) * 0.6),
            arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=8),
            fontsize=10
        )

    # Add model details at the bottom 1/4 of the figure
    model_details = ""
    if config:
        model_details = (
            f"Model: Transformer | Epochs: {len(train_losses)} | Batch Size: {config.batch_size} | "
            f"Learning Rate: {config.learning_rate} | Optimizer: {config.optimizer_type} | "
            f"Layers: {config.num_layers} | Heads: {config.num_heads} | Hidden Dim: {config.hidden_dim} | "
            f"Loss Function: {config.loss_function} | Scheduler: {config.scheduler_type or 'None'} | "
            f"Generated: {get_maputo_timestamp()}"
        )

    plt.subplot(4, 1, 4)
    plt.axis('off')
    plt.text(0.01, 0.5, model_details, wrap=True, fontsize=9)

    # Save with timestamp
    timestamp = get_maputo_timestamp()
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f'training_history_{timestamp}.png'), dpi=300)
    plt.close()

def create_summary_comparison_plot(
    transformer_metrics: List[Tuple[float, float, float, float]],
    baseline_metrics: Dict[str, List[Tuple[float, float, float, float]]],
    results_dir: str,
    config: TrainingConfig
) -> None:
    """
    Creates a comprehensive comparison plot of all models
    """
    plt.figure(figsize=(16, 10))

    # Calculate average metrics for transformer
    avg_transformer = np.mean(transformer_metrics, axis=0)

    # Prepare data for comparison
    models = ['Transformer']
    mae_values = [avg_transformer[0]]
    rmse_values = [avg_transformer[1]]
    r2_values = [avg_transformer[2]]

    # Add baseline models
    for model_name, metrics_list in baseline_metrics.items():
        if metrics_list:
            avg_metrics = np.mean(metrics_list, axis=0)
            models.append(model_name.upper())
            mae_values.append(avg_metrics[0])
            rmse_values.append(avg_metrics[1])
            r2_values.append(avg_metrics[2])

    # Set color scheme
    colors = plt.cm.viridis(np.linspace(0, 0.9, len(models)))

    # Plot MAE (lower is better)
    plt.subplot(2, 2, 1)
    bars = plt.bar(models, mae_values, color=colors)
    plt.title('Mean Absolute Error (lower is better)', fontsize=14)
    plt.ylabel('MAE')
    plt.xticks(rotation=45)

    # Add values on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{height:.2f}', ha='center', va='bottom', fontsize=10)

    # Plot RMSE (lower is better)
    plt.subplot(2, 2, 2)
    bars = plt.bar(models, rmse_values, color=colors)
    plt.title('Root Mean Squared Error (lower is better)', fontsize=14)
    plt.ylabel('RMSE')
    plt.xticks(rotation=45)

    # Add values on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{height:.2f}', ha='center', va='bottom', fontsize=10)

    # Plot R² (higher is better)
    plt.subplot(2, 2, 3)
    bars = plt.bar(models, r2_values, color=colors)
    plt.title('R² Score (higher is better)', fontsize=14)
    plt.ylabel('R²')
    plt.xticks(rotation=45)

    # Add values on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.2f}', ha='center', va='bottom', fontsize=10)

    # Add model configuration info
    plt.subplot(2, 2, 4)
    plt.axis('off')
    model_info = (
        f"MODEL CONFIGURATION\n\n"
        f"Model: Transformer\n"
        f"Layers: {config.num_layers}\n"
        f"Attention Heads: {config.num_heads}\n"
        f"Hidden Dimension: {config.hidden_dim}\n"
        f"Prediction Length: {config.pred_length}\n"
        f"Learning Rate: {config.learning_rate}\n"
        f"Optimizer: {config.optimizer_type}\n"
        f"Loss Function: {config.loss_function}\n"
        f"Improvements: Using {config.scaler_type} scaling"
        f"{', time features' if config.use_time_features else ''}"
        f"{', holiday features' if config.use_holiday_feature else ''}"
    )
    plt.text(0.1, 0.9, model_info, va='top', fontsize=12)

    # Add run timestamp
    timestamp = get_maputo_timestamp()
    plt.figtext(0.5, 0.01, f"Generated: {timestamp}", ha='center', fontsize=10)

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.suptitle('Model Performance Comparison', fontsize=18, fontweight='bold')

    # Save the figure
    plt.savefig(os.path.join(results_dir, f'model_comparison_{timestamp}.png'), dpi=300)
    plt.close()



## Report Generation Module

In [50]:

# =============================================================================
# Report Generation Module
# =============================================================================
def _create_title_page(model, config, timestamp, pdf):
    """
    Creates title page for the traffic prediction report.

    Args:
        model: The trained model
        config: Training configuration
        timestamp: Timestamp for the report
        pdf: PDF object to save the page
    """
    plt.figure(figsize=(12, 8))
    plt.axis('off')

    # Title
    plt.text(0.5, 0.85, "Traffic Prediction Analysis Report",
             fontsize=24, fontweight='bold', ha='center')

    # Subtitle with timestamp
    plt.text(0.5, 0.75, f"Generated on: {timestamp}",
             fontsize=14, ha='center')

    # Model information
    model_info = f"Model: TrafficTransformer"
    if hasattr(model, 'num_layers'):
        model_info += f"\nLayers: {model.num_layers}, Heads: {model.num_heads}"
    if hasattr(model, 'hidden_dim'):
        model_info += f", Hidden Dim: {model.hidden_dim}"
    plt.text(0.5, 0.65, model_info, fontsize=12, ha='center')

    # Configuration highlights
    config_highlights = (
        f"Sequence Length: {config.seq_length}, Prediction Window: {config.pred_length}\n"
        f"Batch Size: {config.batch_size}, Learning Rate: {config.learning_rate}\n"
        f"Optimizer: {config.optimizer_type}, Loss: {config.loss_function}\n"
        f"Using Time Features: {config.use_time_features}, "
        f"Using Holiday Features: {config.use_holiday_feature}"
    )
    plt.text(0.5, 0.55, config_highlights, fontsize=12, ha='center')

    # Footer
    plt.text(0.5, 0.2, "Transformer-Based Traffic Flow Prediction",
             fontsize=16, ha='center', fontstyle='italic')

    pdf.savefig()
    plt.close()

def _create_training_analysis(train_losses, val_losses, pdf):
    """
    Creates training analysis page showing loss curves and convergence patterns.

    Args:
        train_losses: List of training losses per epoch
        val_losses: List of validation losses per epoch
        pdf: PDF object to save the page
    """
    plt.figure(figsize=(12, 8))

    # Plot training and validation loss curves
    epochs = range(1, len(train_losses) + 1)
    plt.subplot(2, 1, 1)
    plt.plot(epochs, train_losses, 'b-', label='Training Loss')
    plt.plot(epochs, val_losses, 'r-', label='Validation Loss')
    plt.title('Training and Validation Loss Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Plot convergence patterns (loss improvement rate)
    plt.subplot(2, 1, 2)
    if len(train_losses) > 5:  # Need enough epochs for moving average
        # Calculate moving average of loss improvement
        window_size = min(5, len(train_losses) // 4)
        train_improvements = [train_losses[i] - train_losses[i+window_size]
                             for i in range(len(train_losses) - window_size)]
        val_improvements = [val_losses[i] - val_losses[i+window_size]
                           for i in range(len(val_losses) - window_size)]

        # Plot improvement rates
        plt.plot(range(window_size + 1, len(train_losses) + 1),
                 train_improvements, 'b--', label='Training Improvement')
        plt.plot(range(window_size + 1, len(val_losses) + 1),
                 val_improvements, 'r--', label='Validation Improvement')
        plt.title('Loss Improvement Over Time (Higher is Better)')
        plt.xlabel('Epochs')
        plt.ylabel('Loss Reduction')
        plt.legend()
        plt.grid(True, alpha=0.3)
    else:
        # Not enough epochs for improvement analysis
        plt.text(0.5, 0.5, "Insufficient epochs for convergence analysis",
                 ha='center', va='center', fontsize=14)

    plt.tight_layout()
    pdf.savefig()
    plt.close()

    # Create additional training insights page if enough data
    if len(train_losses) > 10:
        plt.figure(figsize=(12, 8))

        # Early vs Late convergence
        plt.subplot(2, 2, 1)
        early_epochs = len(train_losses) // 3
        early_improvement = train_losses[0] - train_losses[early_epochs]
        late_improvement = train_losses[early_epochs] - train_losses[-1]

        bars = plt.bar(['Early Phase', 'Late Phase'],
                      [early_improvement, late_improvement])
        plt.title('Loss Improvement: Early vs Late Training')
        plt.ylabel('Loss Reduction')

        # Add values on bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.001,
                    f'{height:.4f}', ha='center', va='bottom')

        # Train-Val Loss Gap
        plt.subplot(2, 2, 2)
        loss_gaps = [val - train for train, val in zip(train_losses, val_losses)]
        plt.plot(epochs, loss_gaps)
        plt.title('Validation-Training Loss Gap')
        plt.xlabel('Epochs')
        plt.ylabel('Gap')
        plt.grid(True, alpha=0.3)

        # Loss distribution
        plt.subplot(2, 2, 3)
        plt.hist(train_losses, bins=10, alpha=0.5, label='Training')
        plt.hist(val_losses, bins=10, alpha=0.5, label='Validation')
        plt.title('Loss Distribution')
        plt.xlabel('Loss Value')
        plt.ylabel('Frequency')
        plt.legend()

        # Stability analysis (loss variance in last 1/3 of training)
        plt.subplot(2, 2, 4)
        stability_start = 2 * len(train_losses) // 3
        train_stability = np.std(train_losses[stability_start:])
        val_stability = np.std(val_losses[stability_start:])

        bars = plt.bar(['Training Stability', 'Validation Stability'],
                      [train_stability, val_stability])
        plt.title('Training Stability (Lower is Better)')
        plt.ylabel('Loss Standard Deviation')

        # Add values on bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.0001,
                    f'{height:.6f}', ha='center', va='bottom')

        plt.tight_layout()
        pdf.savefig()
        plt.close()

def _create_performance_analysis(predictions, actuals, pdf):
    """
    Creates performance analysis page showing actual vs predicted values and error analysis.

    Args:
        predictions: Predicted values
        actuals: Actual values
        pdf: PDF object to save the page
    """
    # Ensure we have proper arrays
    predictions = np.array(predictions).ravel()
    actuals = np.array(actuals).ravel()

    plt.figure(figsize=(12, 10))
    gs = GridSpec(3, 2, figure=plt.gcf())

    # 1. Actual vs Predicted Scatter Plot
    ax1 = plt.subplot(gs[0, 0])
    ax1.scatter(actuals, predictions, alpha=0.5, s=10)

    # Add perfect prediction line
    min_val = min(np.min(actuals), np.min(predictions))
    max_val = max(np.max(actuals), np.max(predictions))
    ax1.plot([min_val, max_val], [min_val, max_val], 'r--')

    ax1.set_title('Actual vs Predicted Values')
    ax1.set_xlabel('Actual')
    ax1.set_ylabel('Predicted')
    ax1.grid(True, alpha=0.3)

    # 2. Error Distribution Histogram
    ax2 = plt.subplot(gs[0, 1])
    errors = predictions - actuals
    ax2.hist(errors, bins=30, alpha=0.7)
    ax2.set_title('Error Distribution')
    ax2.set_xlabel('Prediction Error')
    ax2.set_ylabel('Frequency')
    ax2.grid(True, alpha=0.3)

    # Add mean and std as vertical lines
    mean_error = np.mean(errors)
    std_error = np.std(errors)
    ax2.axvline(mean_error, color='r', linestyle='--', label=f'Mean: {mean_error:.4f}')
    ax2.axvline(mean_error + std_error, color='g', linestyle=':', label=f'Std: {std_error:.4f}')
    ax2.axvline(mean_error - std_error, color='g', linestyle=':')
    ax2.legend()

    # 3. Predicted vs Actual Time Series (sample)
    ax3 = plt.subplot(gs[1, :])
    sample_size = min(200, len(actuals))
    indices = range(sample_size)
    ax3.plot(indices, actuals[:sample_size], 'b-', label='Actual')
    ax3.plot(indices, predictions[:sample_size], 'r-', label='Predicted')
    ax3.set_title('Actual vs Predicted (Sample Time Series)')
    ax3.set_xlabel('Time Step')
    ax3.set_ylabel('Value')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # 4. Residual Plot
    ax4 = plt.subplot(gs[2, 0])
    ax4.scatter(actuals, errors, alpha=0.5, s=10)
    ax4.axhline(y=0, color='r', linestyle='--')
    ax4.set_title('Residual Plot')
    ax4.set_xlabel('Actual Value')
    ax4.set_ylabel('Residual (Error)')
    ax4.grid(True, alpha=0.3)

    # 5. Q-Q Plot for Error Normality
    ax5 = plt.subplot(gs[2, 1])
    stats.probplot(errors, dist="norm", plot=ax5)
    ax5.set_title('Q-Q Plot of Residuals')
    ax5.grid(True, alpha=0.3)

    # Overall metrics text
    mae = mean_absolute_error(actuals, predictions)
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    r2 = r2_score(actuals, predictions)
    mape = 100 * np.mean(np.abs((actuals - predictions) / (actuals + 1e-8)))

    metrics_text = (
        f"MAE: {mae:.4f}\n"
        f"RMSE: {rmse:.4f}\n"
        f"R²: {r2:.4f}\n"
        f"MAPE: {mape:.2f}%"
    )

    plt.figtext(0.5, 0.01, metrics_text, ha="center", fontsize=12,
               bbox={"facecolor":"orange", "alpha":0.2, "pad":5})

    plt.tight_layout(rect=[0, 0.05, 1, 0.95])  # Adjust layout to make room for text
    pdf.savefig()
    plt.close()

def generate_traffic_report(
    model: nn.Module,
    test_loader: DataLoader,
    scaler: Any,
    config: TrainingConfig,
    device: str,
    train_losses: List[float],
    val_losses: List[float],
    fold_metrics: List[Tuple[float, float, float, float]],
    baseline_metrics: Dict[str, List[Tuple[float, float, float, float]]],
    results_dir: str,
    timestamp: Optional[str] = None
) -> str:
    """
    Generate comprehensive report for traffic prediction model analysis with improved error handling

    Args:
        model: Trained model
        test_loader: Test data loader
        scaler: Scaler used for normalization
        config: Training configuration
        device: Device used for model
        train_losses: Training loss history
        val_losses: Validation loss history
        fold_metrics: Metrics for each fold
        baseline_metrics: Metrics for baseline models
        results_dir: Directory to save report
        timestamp: Optional timestamp for the report

    Returns:
        Path to generated report
    """
    timestamp = get_maputo_timestamp() if timestamp is None else timestamp
    pdf_path = os.path.join(results_dir, f'traffic_prediction_report_{timestamp}.pdf')

    # Memory cleanup before generating report
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Set plotting style
    try:
        plt.style.use('seaborn-v0_8')
    except:
        try:
            plt.style.use('seaborn')
        except:
            plt.style.use('default')

    # Set publication-quality figure parameters
    plt.rcParams.update({
        'figure.figsize': (10, 6),
        'font.size': 12,
        'axes.labelsize': 14,
        'axes.titlesize': 16,
        'figure.titlesize': 20,
        'axes.grid': True,
        'grid.alpha': 0.3,
        'lines.linewidth': 2,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight'
    })

    # Utility function for error pages
    def _create_error_page(error_message: str, pdf: PdfPages) -> None:
        """Creates an error page for the PDF when a section fails."""
        plt.figure(figsize=(12, 8))
        plt.axis('off')
        plt.text(0.5, 0.5, f"Error: {error_message}",
                ha='center', va='center', color='red', fontsize=14, wrap=True)
        pdf.savefig()
        plt.close()

    # Collect predictions and attention weights
    model.eval()
    predictions = []
    actuals = []
    attention_weights = []

    try:
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)

                # Store attention weights if available
                if hasattr(model, 'attention_weights') and model.attention_weights is not None:
                    # Handle different attention weight shapes
                    weights = model.attention_weights
                    if isinstance(weights, torch.Tensor):
                        weights = weights.cpu().numpy()
                    attention_weights.append(weights)

                # Store predictions and actuals
                pred = output.cpu().numpy()
                predictions.append(pred)
                actuals.append(target.cpu().numpy())

        predictions = np.concatenate(predictions)
        actuals = np.concatenate(actuals)

        # Reshape and inverse transform if needed
        if len(predictions.shape) > 2:
            predictions = predictions.reshape(-1, predictions.shape[-1])
            actuals = actuals.reshape(-1, actuals.shape[-1])
    except Exception as e:
        print(f"Error collecting predictions: {str(e)}")
        # Create minimal emergency report with available data
        _create_emergency_report(
            predictions or np.array([]),
            actuals or np.array([]),
            train_losses,
            val_losses,
            pdf_path
        )
        return pdf_path

    # Create PDF report with section-by-section error handling
    try:
        with PdfPages(pdf_path) as pdf:
            # 1. Title Page
            try:
                _create_title_page(model, config, timestamp, pdf)
            except Exception as e:
                _create_error_page(f"Error in title page: {str(e)}", pdf)
                print(f"Error in title page: {str(e)}")

            # 2. Training Analysis
            try:
                _create_training_analysis(train_losses, val_losses, pdf)
            except Exception as e:
                _create_error_page(f"Error in training analysis: {str(e)}", pdf)
                print(f"Error in training analysis: {str(e)}")

            # 3. Performance Analysis
            try:
                if len(predictions) > 0 and len(actuals) > 0:
                    _create_performance_analysis(predictions, actuals, pdf)
                else:
                    _create_error_page("Insufficient prediction data for performance analysis", pdf)
            except Exception as e:
                _create_error_page(f"Error in performance analysis: {str(e)}", pdf)
                print(f"Error in performance analysis: {str(e)}")

            # 4. Attention Analysis
            try:
                if attention_weights:
                    _create_attention_analysis(attention_weights, config, pdf)
                else:
                    _create_error_page("No attention weights available for analysis", pdf)
            except Exception as e:
                _create_error_page(f"Error in attention analysis: {str(e)}", pdf)
                print(f"Error in attention analysis: {str(e)}")

            # 5. Cross Validation Analysis
            try:
                if fold_metrics:
                    _create_cross_validation_analysis(fold_metrics, pdf)
                else:
                    _create_error_page("No fold metrics available for cross-validation analysis", pdf)
            except Exception as e:
                _create_error_page(f"Error in cross validation analysis: {str(e)}", pdf)
                print(f"Error in cross validation analysis: {str(e)}")

            # 6. Baseline Comparison
            try:
                if fold_metrics and any(metrics for metrics in baseline_metrics.values()):
                    _create_baseline_comparison(baseline_metrics, fold_metrics, pdf)
                else:
                    _create_error_page("Insufficient data for baseline comparison", pdf)
            except Exception as e:
                _create_error_page(f"Error in baseline comparison: {str(e)}", pdf)
                print(f"Error in baseline comparison: {str(e)}")

            # 7. Model Configuration
            try:
                _create_config_summary(config, pdf)
            except Exception as e:
                _create_error_page(f"Error in config summary: {str(e)}", pdf)
                print(f"Error in config summary: {str(e)}")

            # 8. Summary Statistics
            try:
                _create_summary_statistics(
                    predictions, actuals, train_losses, val_losses,
                    fold_metrics, baseline_metrics, pdf
                )
            except Exception as e:
                _create_error_page(f"Error in summary statistics: {str(e)}", pdf)
                print(f"Error in summary statistics: {str(e)}")

    except Exception as e:
        print(f"Error generating report: {str(e)}")
        # Create minimal emergency report
        _create_emergency_report(predictions, actuals, train_losses, val_losses, pdf_path)

    return pdf_path


def _create_attention_analysis(
    attention_weights: List[np.ndarray],
    config: TrainingConfig,
    pdf: PdfPages
) -> None:
    """Creates attention analysis visualizations with improved error handling."""
    plt.figure(figsize=(12, 8))

    # Guard against empty attention weights
    if not attention_weights:
        plt.text(0.5, 0.5, "No attention weights available",
                 ha='center', va='center', fontsize=14)
        pdf.savefig()
        plt.close()
        return

    try:
        # Handle different possible shapes of attention weights
        sample_weights = attention_weights[0]
        attention_array = np.array(attention_weights)

        # Detect shape and process accordingly
        if len(attention_array.shape) == 4:  # [batch, heads, seq, seq]
            attention_mean = np.mean(attention_array, axis=(0, 1))  # Average across batch and heads
        elif len(attention_array.shape) == 3:  # [batch, seq, seq]
            attention_mean = np.mean(attention_array, axis=0)  # Average across batch
        elif len(attention_array.shape) == 2:  # Already [seq, seq]
            attention_mean = attention_array
        else:
            # If we have a list of tensors with different shapes
            reshaped_weights = []
            for weights in attention_weights:
                if hasattr(weights, 'shape'):
                    if len(weights.shape) == 3:  # [batch, seq, seq]
                        weights = np.mean(weights, axis=0)
                    elif len(weights.shape) > 3:  # More dimensions than expected
                        weights = np.mean(weights, axis=tuple(range(len(weights.shape)-2)))
                reshaped_weights.append(weights)

            attention_mean = np.mean(reshaped_weights, axis=0)

        # Create heatmap
        sns.heatmap(attention_mean, cmap="viridis", annot=False)
        plt.title('Average Attention Weights')
        plt.xlabel('Key Position')
        plt.ylabel('Query Position')

    except Exception as e:
        # Create a fallback visualization with error message
        plt.clf()  # Clear the figure
        plt.text(0.5, 0.5, f"Error visualizing attention weights: {str(e)}\n"
                           f"Shape info: {[w.shape if hasattr(w, 'shape') else type(w) for w in attention_weights[:3]]}...",
                 ha='center', va='center', wrap=True)
        plt.title('Attention Visualization Error')

    pdf.savefig()
    plt.close()


def _create_cross_validation_analysis(
    fold_metrics: List[Tuple[float, float, float, float]],
    pdf: PdfPages
) -> None:
    """Creates cross validation analysis visualizations."""
    metrics_names = ['MAE', 'RMSE', 'R²', 'MAPE']
    metrics_values = np.array(fold_metrics)

    plt.figure(figsize=(12, 6))
    for i, metric in enumerate(metrics_names):
        plt.subplot(2, 2, i+1)
        plt.boxplot(metrics_values[:, i])
        plt.title(f'{metric} Across Folds')
        plt.grid(True, alpha=0.3)

    plt.tight_layout()
    pdf.savefig()
    plt.close()


def _create_baseline_comparison(
    baseline_metrics: Dict[str, List[Tuple[float, float, float, float]]],
    fold_metrics: List[Tuple[float, float, float, float]],
    pdf: PdfPages
) -> None:
    """Creates baseline comparison visualizations."""
    transformer_metrics = np.mean(fold_metrics, axis=0)
    metrics_names = ['MAE', 'RMSE', 'R²', 'MAPE']

    # Prepare data for plotting
    models = ['Transformer'] + list(baseline_metrics.keys())
    metrics_data = [transformer_metrics] + [np.mean(metrics, axis=0) if metrics else [np.nan]*4
                                           for metrics in baseline_metrics.values()]

    # Create comparison plots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    for i, (metric, ax) in enumerate(zip(metrics_names, axes.ravel())):
        metric_values = [data[i] for data in metrics_data]
        ax.bar(models, metric_values)
        ax.set_title(f'{metric} Comparison')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    pdf.savefig()
    plt.close()


def _create_config_summary(config: TrainingConfig, pdf: PdfPages) -> None:
    """Creates configuration summary page."""
    plt.figure(figsize=(12, 8))
    plt.axis('off')

    config_text = "Model Configuration:\n\n"
    for key, value in vars(config).items():
        if not key.startswith('_') and key not in ['input_dir', 'output_dir', 'model_dir', 'results_dir']:  # Skip directories for brevity
            config_text += f"{key}: {value}\n"

    plt.text(0.1, 0.9, config_text, fontsize=10, va='top')
    pdf.savefig()
    plt.close()


def _create_summary_statistics(
    predictions: np.ndarray,
    actuals: np.ndarray,
    train_losses: List[float],
    val_losses: List[float],
    fold_metrics: List[Tuple[float, float, float, float]],
    baseline_metrics: Dict[str, List[Tuple[float, float, float, float]]],
    pdf: PdfPages
) -> None:
    """Creates summary statistics page."""
    plt.figure(figsize=(12, 8))
    plt.axis('off')

    # Calculate overall metrics
    mae = mean_absolute_error(actuals.ravel(), predictions.ravel())
    rmse = np.sqrt(mean_squared_error(actuals.ravel(), predictions.ravel()))
    r2 = r2_score(actuals.ravel(), predictions.ravel())

    summary_text = f"""
    Overall Model Performance:

    Mean Absolute Error: {mae:.4f}
    Root Mean Squared Error: {rmse:.4f}
    R² Score: {r2:.4f}

    Training Summary:
    Initial Training Loss: {train_losses[0]:.6f}
    Final Training Loss: {train_losses[-1]:.6f}
    Loss Improvement: {train_losses[0] - train_losses[-1]:.6f}

    Cross-Validation Summary:
    Number of Folds: {len(fold_metrics)}
    Average MAE across folds: {np.mean([m[0] for m in fold_metrics]):.4f}
    Average RMSE across folds: {np.mean([m[1] for m in fold_metrics]):.4f}

    Baseline Comparison:
    """

    for model_name, metrics in baseline_metrics.items():
        if metrics:  # Check if metrics list is not empty
            avg_metrics = np.mean(metrics, axis=0)
            summary_text += f"\n{model_name.upper()} - MAE: {avg_metrics[0]:.4f}, RMSE: {avg_metrics[1]:.4f}"
        else:
            summary_text += f"\n{model_name.upper()} - No metrics available"

    plt.text(0.1, 0.9, summary_text, fontsize=12, va='top')
    pdf.savefig()
    plt.close()


def _create_emergency_report(
    predictions: np.ndarray,
    actuals: np.ndarray,
    train_losses: List[float],
    val_losses: List[float],
    pdf_path: str
) -> None:
    """Creates a minimal emergency report if the full report fails."""
    try:
        with PdfPages(pdf_path) as pdf:
            plt.figure(figsize=(12, 8))
            plt.axis('off')

            mae = mean_absolute_error(actuals.ravel(), predictions.ravel())
            rmse = np.sqrt(mean_squared_error(actuals.ravel(), predictions.ravel()))
            r2 = r2_score(actuals.ravel(), predictions.ravel())

            emergency_text = f"""
            Emergency Report (Error in full report generation)

            Basic Metrics:
            MAE: {mae:.4f}
            RMSE: {rmse:.4f}
            R² Score: {r2:.4f}

            Final Losses:
            Training: {train_losses[-1]:.6f}
            Validation: {val_losses[-1]:.6f}
            """
            plt.text(0.1, 0.9, emergency_text, fontsize=12, va='top')
            pdf.savefig()
            plt.close()
    except Exception as e2:
        warnings.warn(f"Emergency report also failed: {str(e2)}")



## Main Execution

In [51]:

# =============================================================================
# Main Execution
# =============================================================================

def main():
    """Main execution function with enhanced plotting and reporting"""
    # --- Load Configuration ---
    config_path = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/config.yaml'
    if os.path.exists(config_path):
        config = load_config(config_path)
        print("Configuration loaded from config.yaml")
    else:
        print("config.yaml not found, using default parameters.")
        config = TrainingConfig(base_output_dir='/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction')

    # Print initial GPU info
    if torch.cuda.is_available():
        print("\n=== Initial GPU Information ===")
        gpu_info = get_gpu_memory_info()
        for gpu_id, info in gpu_info.items():
            if isinstance(info, dict) and 'error' not in info:
                print(f"{gpu_id.upper()}: Total {info['total_memory_GB']:.2f} GB, Free {info['free_memory_GB']:.2f} GB")
        print("===============================\n")

    # --- Setup Directories ---
    input_dir, output_dir, model_dir, results_dir = setup_directories(config)

    # --- Mount Google Drive if in Colab ---
    if IN_COLAB:
        try:
            drive.mount('/content/drive')
            print("Google Drive mounted successfully")
        except:
            warnings.warn("Failed to mount Google Drive, using local directories")

    # --- Load and Preprocess Data ---
    try:
        df = pd.read_csv(os.path.join(input_dir, 'METR-LA.csv'), index_col=0, parse_dates=True)
    except FileNotFoundError:
        raise FileNotFoundError(f"METR-LA.csv not found in {input_dir}, please place the dataset file there")

    data_normalized, timestamps, scaler, num_features = prepare_data(df, config)

    # --- Adjacency Matrix Preparation ---
    if config.use_spatial_features and config.use_gnn_pre_transformer:
        if not TORCH_GEOMETRIC_AVAILABLE:
            raise ImportError("PyTorch Geometric is required for GNN pre-transformer but not available")

        dataset_instance = TrafficDataset(data_normalized, timestamps, config.seq_length, config.pred_length, config)
        adjacency_matrix = dataset_instance.get_adjacency_matrix()
    else:
        adjacency_matrix = None

    # --- Model Parameters ---
    model_params = {
        'input_dim': num_features +
                    (4 if config.use_time_features else 0) +
                    (1 if config.use_holiday_feature else 0) +
                    (1 if config.use_weather_feature else 0) +
                    (num_features * config.num_lags if config.use_lagged_features else 0) +
                    (config.spatial_feature_dim if config.use_spatial_features and not config.use_gnn_pre_transformer else 0),
        'hidden_dim': config.hidden_dim,
        'num_layers': config.num_layers,
        'num_heads': config.num_heads,
        'num_features': num_features,  # Pass explicit number of features
        'dropout': config.dropout,
        'ff_dim_multiplier': config.ff_dim_multiplier,
        'activation': config.activation,
        'decoder_type': config.decoder_type,
        'use_gnn_pre_transformer': config.use_gnn_pre_transformer,
        'spatial_feature_dim': config.spatial_feature_dim,
        'gnn_type': config.gnn_type,
        'pred_len': config.pred_length,
    }

    # --- Set Device ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # *** Add dimension compatibility check here before model creation ***
    model_params, config = ensure_compatible_dimensions(model_params, config)

    # Automatically find optimal batch size if enabled
    if config.find_optimal_batch_size and device == 'cuda' and torch.cuda.is_available():
        print("\n=== Finding Optimal Batch Size ===")
        # Create a dummy dataset and model for batch size testing
        dummy_input = torch.randn(1, config.seq_length, model_params['input_dim'])
        dummy_target = torch.randn(1, config.pred_length, num_features)

        # Create a temporary model instance for testing
        # *** Add dimension compatibility check before model creation ***
        temp_model_params, config = ensure_compatible_dimensions(model_params.copy(), config)
        temp_model = TrafficTransformer(**temp_model_params)

        # Find optimal batch size
        optimal_batch_size = find_optimal_batch_size(
            temp_model,
            dummy_input,
            dummy_target,
            max_batch_size=2048,  # Upper limit to test
            start_batch=32        # Starting test size
        )

        print(f"Optimal batch size for GPU memory: {optimal_batch_size}")

        # Update configuration with optimal batch size
        config.batch_size = optimal_batch_size

        # Clean up
        del temp_model, dummy_input, dummy_target
        torch.cuda.empty_cache()
        gc.collect()

    # --- Optuna Hyperparameter Optimization ---
    if OPTUNA_AVAILABLE and config.optuna_trials is not None and config.optuna_trials > 0:
        # Define the objective function with proper config access
        def objective(trial):
            # We need to declare config as nonlocal since it's from the outer scope
            nonlocal config

            # Define hyperparameter search space
            num_heads = trial.suggest_categorical('num_heads', [4, 8, 16])

            # Ensure hidden_dim is both divisible by num_heads and is even
            hidden_dim_base = trial.suggest_int('hidden_dim', 64, 256)
            # Adjust to ensure divisibility
            hidden_dim = (hidden_dim_base // num_heads) * num_heads
            if hidden_dim % 2 != 0:  # If not even, make it even
                hidden_dim = hidden_dim + num_heads if hidden_dim % num_heads == 0 else ((hidden_dim // num_heads) + 1) * num_heads

            # Create a copy of the config to avoid modifying the original
            trial_config = TrainingConfig(
                base_output_dir=config.base_output_dir,
                hidden_dim=hidden_dim,
                num_layers=trial.suggest_int('num_layers', 2, 6),
                num_heads=num_heads,
                dropout=trial.suggest_float('dropout', 0.0, 0.5),
                learning_rate=trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True),
                decoder_type=trial.suggest_categorical('decoder_type', ['linear', 'mlp'])
            )

            # Copy important settings from the original config
            trial_config.seq_length = config.seq_length
            trial_config.pred_length = config.pred_length
            trial_config.use_time_features = config.use_time_features
            trial_config.use_holiday_feature = config.use_holiday_feature
            trial_config.use_weather_feature = config.use_weather_feature
            trial_config.use_lagged_features = config.use_lagged_features
            trial_config.use_spatial_features = config.use_spatial_features
            trial_config.use_gnn_pre_transformer = config.use_gnn_pre_transformer
            trial_config.num_workers = config.num_workers
            trial_config.batch_size = config.batch_size

            # CRITICAL: Copy directory paths from main config
            trial_config.input_dir = config.input_dir
            trial_config.output_dir = config.output_dir
            trial_config.model_dir = config.model_dir
            trial_config.results_dir = config.results_dir

            print(f"Trial with hidden_dim: {hidden_dim}, num_heads: {num_heads}, num_layers: {trial_config.num_layers}")
            print(f"Trial config directories: model_dir={trial_config.model_dir}")

            # Copy model parameters but with trial values
            trial_model_params = model_params.copy()
            trial_model_params['hidden_dim'] = hidden_dim
            trial_model_params['num_layers'] = trial_config.num_layers
            trial_model_params['num_heads'] = num_heads
            trial_model_params['dropout'] = trial_config.dropout
            trial_model_params['decoder_type'] = trial_config.decoder_type

            # Ensure dimensions are compatible
            trial_model_params, trial_config = ensure_compatible_dimensions(trial_model_params, trial_config)

            # Double-check the constraint is satisfied
            assert trial_model_params['hidden_dim'] % trial_model_params['num_heads'] == 0, \
                f"Hidden dimension {trial_model_params['hidden_dim']} must be divisible by number of heads {trial_model_params['num_heads']}"

            try:
                # Create model with trial parameters
                model = TrafficTransformer(**trial_model_params).to(device)

                # Initialize optimizer and scheduler
                optimizer = optim.AdamW(model.parameters(), lr=trial_config.learning_rate)
                scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, patience=trial_config.patience // 2, factor=0.5
                )

                # Choose criterion based on loss function
                if trial_config.loss_function == 'mse':
                    criterion = nn.MSELoss()
                elif trial_config.loss_function == 'mae':
                    criterion = nn.L1Loss()
                elif trial_config.loss_function == 'quantile':
                    criterion = lambda output, target: quantile_loss(output, target, trial_config.quantiles)
                elif trial_config.loss_function == 'hybrid':
                    criterion = hybrid_loss
                else:
                    criterion = nn.MSELoss()

                # Time Series Cross-Validation (Single Fold for Optuna speed)
                tscv_optuna = TimeSeriesSplit(n_splits=2)
                fold_val_losses = []

                for fold_optuna, (train_idx_optuna, val_idx_optuna) in enumerate(tscv_optuna.split(data_normalized)):
                    train_data_optuna = data_normalized[train_idx_optuna]
                    val_data_optuna = data_normalized[val_idx_optuna]
                    train_times_optuna = timestamps[train_idx_optuna]
                    val_times_optuna = timestamps[val_idx_optuna]

                    train_dataset_optuna = TrafficDataset(
                        train_data_optuna, train_times_optuna, trial_config.seq_length, trial_config.pred_length, trial_config
                    )
                    val_dataset_optuna = TrafficDataset(
                        val_data_optuna, val_times_optuna, trial_config.seq_length, trial_config.pred_length, trial_config
                    )

                    # Use smaller batch size for trials to avoid memory issues
                    trial_batch_size = min(trial_config.batch_size, 64)  # Limiting batch size for trials

                    train_loader_optuna = DataLoader(
                        train_dataset_optuna, batch_size=trial_batch_size, shuffle=True,
                        num_workers=min(trial_config.num_workers, 1),  # Reduce workers for trials
                        pin_memory=trial_config.pin_memory
                    )
                    val_loader_optuna = DataLoader(
                        val_dataset_optuna, batch_size=trial_batch_size, shuffle=False,
                        num_workers=min(trial_config.num_workers, 1),  # Reduce workers for trials
                        pin_memory=trial_config.pin_memory
                    )

                    # Train the model with fewer epochs for speed
                    trial_config.num_epochs = min(trial_config.num_epochs, 10)  # Limit epochs for trials

                    # Train the model
                    trained_model_optuna, train_losses_optuna, val_losses_optuna = train_model(
                        model, train_loader_optuna, val_loader_optuna, optimizer, scheduler,
                        criterion, trial_config, device, adjacency_matrix
                    )

                    # Evaluate the model
                    avg_val_loss_optuna, _ = evaluate_model(
                        trained_model_optuna, val_loader_optuna, criterion, device, trial_config, adjacency_matrix
                    )
                    fold_val_losses.append(avg_val_loss_optuna)

                mean_val_loss = np.mean(fold_val_losses)
                print(f"Trial completed successfully with mean validation loss: {mean_val_loss:.6f}")
                return mean_val_loss

            except Exception as e:
                print(f"Trial failed with error: {str(e)}")
                # Return a high value to indicate failure
                return float('inf')

        # Create and run Optuna study
        study = optuna.create_study(direction='minimize')
        print(f"Starting Optuna with {config.optuna_trials} trials")

        try:
            study.optimize(objective, n_trials=config.optuna_trials)

            # Check if we have valid trials
            if study.trials:
                # Get best parameters from completed trials
                best_params = study.best_params
                print(f"Optuna best hyperparameters: {best_params}")

                # Update config with best parameters
                for param, value in best_params.items():
                    setattr(config, param, value)

                # Update model parameters with best parameters
                model_params['hidden_dim'] = config.hidden_dim
                model_params['num_layers'] = config.num_layers
                model_params['num_heads'] = config.num_heads
                model_params['dropout'] = config.dropout
                model_params['decoder_type'] = config.decoder_type
            else:
                print("No successful Optuna trials completed. Using original parameters.")

        except Exception as e:
            print(f"Optuna optimization failed: {str(e)}")
            print("Continuing with original parameters...")

        # Ensure dimensions are compatible
        model_params, config = ensure_compatible_dimensions(model_params, config)
    else:
        if not OPTUNA_AVAILABLE:
            warnings.warn("Optuna not available, skipping hyperparameter tuning")
        else:
            print("Optuna hyperparameter tuning disabled (optuna_trials is None or 0)")

    # --- Time Series Cross-Validation ---
    tscv = TimeSeriesSplit(n_splits=5)
    fold_metrics = []
    baseline_metrics = {'arima': [], 'exp_smoothing': [], 'lstm': []}

    # Start GPU memory monitoring - non-contextmanager version
    monitor_stop_flag = start_gpu_memory_monitor(config, interval=15)

    try:
        # Iterate through time series folds
        for fold, (train_idx, test_idx) in enumerate(tscv.split(data_normalized)):
            print(f"\n=== Fold {fold+1} ===")
            print(f"Config: {config.scaler_type}, {config.optimizer_type}, {config.loss_function}, {config.pred_length}-step ahead")

            # Split data
            train_data = data_normalized[train_idx]
            test_data = data_normalized[test_idx]
            train_times = timestamps[train_idx]
            test_times = timestamps[test_idx]

            # Create datasets
            train_dataset = TrafficDataset(train_data, train_times, config.seq_length, config.pred_length, config)
            test_dataset = TrafficDataset(test_data, test_times, config.seq_length, config.pred_length, config)

            print(f"\nDataset created - Train: {len(train_dataset)} samples, Test: {len(test_dataset)} samples")


            # Enhanced DataLoader configuration with workers and pinned memory
            train_loader = DataLoader(
                train_dataset,
                batch_size=config.batch_size,
                shuffle=True,
                num_workers=config.num_workers,
                pin_memory=config.pin_memory,
                prefetch_factor=2  # Prefetch 2 batches per worker
            )
            test_loader = DataLoader(
                test_dataset,
                batch_size=config.batch_size,
                shuffle=False,
                num_workers=config.num_workers,
                pin_memory=config.pin_memory
            )

            # Double-check dimension compatibility before creating model
            model_params, config = ensure_compatible_dimensions(model_params, config)

            # Initialize model
            model = TrafficTransformer(**model_params).to(device)

            # Initialize optimizer
            if config.optimizer_type == 'adam':
                optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
            elif config.optimizer_type == 'adamw':
                optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate)
            else:
                warnings.warn(f"Invalid optimizer type: {config.optimizer_type}, using AdamW")
                optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate)

            # Initialize loss function
            if config.loss_function == 'mse':
                criterion = nn.MSELoss()
            elif config.loss_function == 'mae':
                criterion = nn.L1Loss()
            elif config.loss_function == 'quantile':
                criterion = lambda output, target: quantile_loss(output, target, config.quantiles)
            elif config.loss_function == 'hybrid':
                criterion = hybrid_loss
            else:
                warnings.warn(f"Invalid loss function: {config.loss_function}, using MSELoss")
                criterion = nn.MSELoss()

            # Initialize scheduler
            if config.scheduler_type == 'plateau':
                scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, patience=config.scheduler_patience, factor=config.scheduler_factor
                )
            elif config.scheduler_type == 'cosine':
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.num_epochs)
            elif config.scheduler_type == 'step':
                scheduler = optim.lr_scheduler.StepLR(
                    optimizer, step_size=config.step_scheduler_step_size, gamma=config.step_scheduler_gamma
                )
            elif config.scheduler_type == 'cosine_warmup':
                scheduler = CosineWarmupLR(
                    optimizer,
                    warmup_epochs=config.warmup_epochs,
                    total_epochs=config.num_epochs,
                    base_lr=config.learning_rate
                )
            elif config.scheduler_type is None:
                scheduler = None
            else:
                warnings.warn(f"Invalid scheduler type: {config.scheduler_type}, using ReduceLROnPlateau")
                scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, patience=config.scheduler_patience, factor=config.scheduler_factor
                )

            # Explicitly run garbage collection and clear CUDA cache before training
            gc.collect()
            torch.cuda.empty_cache()

            # Train model
            trained_model, fold_train_losses, fold_val_losses = train_model(
                model, train_loader, test_loader, optimizer, scheduler,
                criterion, config, device, adjacency_matrix
            )

            # Evaluate model
            test_loss, metrics = evaluate_model(
                trained_model, test_loader, criterion, device, config, adjacency_matrix
            )
            fold_metrics.append(metrics)
            mae, rmse, r2, mape = metrics

            print(f'Fold {fold+1} Test Metrics: MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.2f}, MAPE: {mape:.2f}%')

            # Save predictions and plots
            predictions, actuals = predict(trained_model, test_loader, scaler, device, adjacency_matrix)

            # MODIFIED: Pass config to plotting functions for enhanced details
            for sensor_idx in range(min(3, actuals.shape[1])):  # Plot first 3 sensors
                plot_predictions_vs_actual(
                    actuals, predictions, sensor_idx, fold,
                    results_dir, config.pred_length, config  # Pass config here
                )

            plot_attention_weights(
                trained_model, config.seq_length,
                results_dir, config  # Pass config here
            )

            # Plot training history with enhanced details
            plot_training_history(
                fold_train_losses, fold_val_losses,
                results_dir, config  # Pass config here
            )

            # Train and evaluate baseline models
            print("\nTraining baseline models (optimized for faster execution)...")
            fold_baseline_metrics = train_baseline_models(
                train_data,
                test_data,
                config,
                device,
                timestamps_train=train_times,
                timestamps_test=test_times
            )

            # Add fold baseline metrics to overall baseline metrics
            for model_name, metrics_list in fold_baseline_metrics.items():
                if metrics_list:  # Only append if there are metrics
                    baseline_metrics[model_name].extend(metrics_list)

    except Exception as e:
        print(f"Error during training: {e}")
        import traceback
        traceback.print_exc()

    finally:
        # Stop GPU monitoring
        stop_gpu_memory_monitor(monitor_stop_flag)

    # --- Calculate and print average metrics ---
    if fold_metrics:
        avg_mae = np.mean([m[0] for m in fold_metrics])
        avg_rmse = np.mean([m[1] for m in fold_metrics])
        avg_r2 = np.mean([m[2] for m in fold_metrics])
        avg_mape = np.nanmean([m[3] for m in fold_metrics])  # Use nanmean to handle NaN values

        print(f"\n=== Average Metrics Across Folds ===")
        print(f"Transformer MAE: {avg_mae:.2f}, RMSE: {avg_rmse:.2f}, R²: {avg_r2:.2f}, MAPE: {avg_mape:.2f}%")

        # --- Print baseline metrics ---
        print("\n=== Average Baseline Metrics Across Folds ===")
        for model_name, metrics_list in baseline_metrics.items():
            if metrics_list:  # Only calculate averages if there are metrics
                avg_baseline_mae = np.mean([m[0] for m in metrics_list])
                avg_baseline_rmse = np.mean([m[1] for m in metrics_list])
                avg_baseline_r2 = np.mean([m[2] for m in metrics_list])
                avg_baseline_mape = np.nanmean([m[3] for m in metrics_list])
                print(f"Model: {model_name.upper()}")
                print(f"  MAE: {avg_baseline_mae:.2f}, RMSE: {avg_baseline_rmse:.2f}, R²: {avg_baseline_r2:.2f}, MAPE: {avg_baseline_mape:.2f}%")

        # --- Generate final report ---
        try:
            print("\nGenerating comprehensive traffic prediction report...")
            report_path = generate_traffic_report(
                model=trained_model,
                test_loader=test_loader,
                scaler=scaler,
                config=config,
                device=device,
                train_losses=fold_train_losses,
                val_losses=fold_val_losses,
                fold_metrics=fold_metrics,
                baseline_metrics=baseline_metrics,
                results_dir=results_dir,
                timestamp=get_maputo_timestamp()
            )

            if report_path:
                print(f"\nDetailed analysis report generated at: {report_path}")
            else:
                print("\nWarning: Failed to generate detailed analysis report")
        except Exception as e:
            print(f"Error generating final report: {e}")

    else:
        print("\nNo fold metrics available - training may have failed")

    # Create summary plot with all model comparisons
    try:
        if fold_metrics and any(metrics for metrics in baseline_metrics.values()):
            create_summary_comparison_plot(
                fold_metrics,
                baseline_metrics,
                results_dir,
                config
            )
    except Exception as e:
        print(f"Error creating summary comparison plot: {e}")

    print("\n--- Training and Evaluation Complete ---")


if __name__ == "__main__":
    # Configure PyTorch to optimize for performance
    torch.backends.cudnn.benchmark = True  # Enable cudnn auto-tuner

    # Set any environment variables for performance
    os.environ['OMP_NUM_THREADS'] = str(min(os.cpu_count(), 8))  # Limit OpenMP threads

    try:
        # Run main function with error handling
        main()
    except Exception as e:
        print(f"Error in main execution: {e}")
        import traceback
        traceback.print_exc()

    # Final cleanup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("\n=== Final GPU Memory Usage ===")
        gpu_info = get_gpu_memory_info()
        for gpu_id, info in gpu_info.items():
            if isinstance(info, dict) and 'error' not in info:
                print(f"{gpu_id.upper()}: Used {info['allocated_memory_GB']:.2f}/{info['total_memory_GB']:.2f} GB "
                      f"({info['utilization_pct']:.1f}%)")
        print("===============================")

config.yaml not found, using default parameters.

=== Initial GPU Information ===
GPU_0: Total 15.83 GB, Free 15.83 GB

Error in main execution: METR-LA.csv not found in /content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformers_Input, please place the dataset file there

=== Final GPU Memory Usage ===
GPU_0: Used 0.00/15.83 GB (0.0%)


<ipython-input-51-4ef9dfadfbec>:34: UserWarning: Failed to mount Google Drive, using local directories
  warnings.warn("Failed to mount Google Drive, using local directories")
Traceback (most recent call last):
  File "<ipython-input-51-4ef9dfadfbec>", line 38, in main
    df = pd.read_csv(os.path.join(input_dir, 'METR-LA.csv'), index_col=0, parse_dates=True)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.e